# Thermocouple Calibration — April 10, 2026

**Source file:** `log_20260410_145703.csv`

> **Warning:** Another calibration run is required before these coefficients should be used for the `T`-type probes (`TTEST`, `TFO`, `TTI`, `TTO`, `TMI`). During this April 10 run the firmware had the MAX31856 channels configured as `K`-type, so the `T`-type thermocouples were decoded with the wrong thermocouple table. `THM` and `THI` are still `K`-type probes, but this notebook should be treated as a record of the flawed first pass rather than the final calibration.

Two calibration events were recorded:

- **Dip 1** (~t = 188 750 – 189 070 s): `TTO`, `TMI`, `TFO`, and `TTI` were dipped sequentially into liquid nitrogen.  All four reached a stable LN plateau (~−177 °C raw).
- **Dip 2** (~t = 194 100 – 194 620 s): `THM` and `THI` were dipped but never fully reached LN saturation temperature (THM min ≈ −172 °C, THI min ≈ −147 °C). A two-point LN calibration is **not applicable** to these two channels.

**Calibration model** (two-point affine):
$$T_\text{cal} = g \cdot T_\text{raw} + b$$
with two known anchor pairs:
- Room anchor: $T_\text{true} = 68.5\ °\text{F} = 20.28\ °\text{C}$, $T_\text{raw}$ = pre-dip median.
- LN anchor: $T_\text{true} = -195.8\ °\text{C}$ (LN₂ boiling point at 1 atm), $T_\text{raw}$ = stable plateau median.

For `THM` and `THI` a single-point offset (gain = 1, only room-temperature offset) is reported instead.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data' / 'raw').exists() and (candidate / 'analysis' / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Could not locate repo root.')


repo_root = find_repo_root()
analysis_src = repo_root / 'analysis' / 'src'
if str(analysis_src) not in sys.path:
    sys.path.insert(0, str(analysis_src))

# ── constants ──────────────────────────────────────────────────────────────
LN2_BP_C = -195.8          # LN₂ boiling point at 1 atm [°C]
ROOM_TRUE_F = 68.5         # known lab ambient [°F]
ROOM_TRUE_C = (ROOM_TRUE_F - 32.0) * 5.0 / 9.0

# time windows (absolute, seconds-of-day from the logger)
ROOM_WINDOW = (188_424, 188_745)   # pre-dip stable period for all TCs

# stable LN plateau windows (dip 1)
LN_WINDOWS = {
    'TTO_C':  (188_771, 188_791),
    'TMI_C':  (188_840, 188_867),
    'TFO_C':  (188_909, 188_940),
    'TTI_C':  (189_023, 189_041),
}

# channels that could NOT be calibrated at LN (dip 2 — not fully immersed)
NO_LN_CHANNELS = ['THM_C', 'THI_C']

TC_CALIBRATION_PATH = repo_root / 'data' / 'processed' / 'calibration' / 'TC_calibration_20260420.csv'
CAL_LOG = repo_root / 'data' / 'raw' / 'calibration' / 'log_20260410_145703.csv'
TC_COLS  = ['TTEST_C', 'TFO_C', 'TTI_C', 'TTO_C', 'TMI_C', 'THM_C', 'THI_C']
COLORS   = dict(zip(TC_COLS, ['C0','C1','C2','C3','C4','C5','C6']))

In [2]:
df = pd.read_csv(CAL_LOG)
df['time_s'] = pd.to_numeric(df['time_s'], errors='coerce')
for col in TC_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=['time_s']).sort_values('time_s').reset_index(drop=True)
df['t_rel_min'] = (df['time_s'] - df['time_s'].iloc[0]) / 60.0

print(f"Rows: {len(df)}")
print(f"Time range: {df['time_s'].iloc[0]:.0f} – {df['time_s'].iloc[-1]:.0f} s")
print(f"Duration: {df['t_rel_min'].iloc[-1]:.1f} min")

Rows: 3591
Time range: 188425 – 194944 s
Duration: 108.7 min


## Overview — raw TC temperatures

In [3]:
fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)

for col in TC_COLS:
    mask = df[col].notna()
    ax.plot(df.loc[mask, 't_rel_min'], df.loc[mask, col],
            lw=1.0, label=col.replace('_C', ''), color=COLORS[col])

# shade calibration windows
t0 = df['time_s'].iloc[0]
room_span = [(ROOM_WINDOW[0] - t0)/60, (ROOM_WINDOW[1] - t0)/60]
ax.axvspan(*room_span, alpha=0.12, color='green', label='Room anchor window')

for tc, (t_lo, t_hi) in LN_WINDOWS.items():
    ax.axvspan((t_lo - t0)/60, (t_hi - t0)/60, alpha=0.10, color='steelblue')

dip2_span = [(194_100 - t0)/60, (194_620 - t0)/60]
ax.axvspan(*dip2_span, alpha=0.10, color='orange', label='Dip 2 (THM / THI — partial)')

ax.axhline(LN2_BP_C, color='navy', lw=1.0, ls='--', alpha=0.7, label=f'LN₂ boiling point ({LN2_BP_C} °C)')
ax.axhline(ROOM_TRUE_C, color='green', lw=1.0, ls='--', alpha=0.7, label=f'Room reference ({ROOM_TRUE_F:.1f} °F / {ROOM_TRUE_C:.3f} °C)')

ax.set_xlabel('Time from log start [min]')
ax.set_ylabel('Raw temperature [°C]')
ax.set_title('TC calibration session — log_20260410_145703.csv')
ax.legend(loc='lower right', fontsize=8, ncol=2)
plt.show()

/tmp/ipykernel_1222632/1599018127.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Dip detail — Dip 1 (TTO, TMI, TFO, TTI)

In [4]:
dip1_cols = ['TTO_C', 'TMI_C', 'TFO_C', 'TTI_C']
dip1_mask = (df['time_s'] >= 188_600) & (df['time_s'] <= 189_200)
dip1 = df.loc[dip1_mask].copy()

fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
for col in dip1_cols:
    mask = dip1[col].notna()
    ax.plot(dip1.loc[mask, 't_rel_min'], dip1.loc[mask, col],
            lw=1.2, label=col.replace('_C', ''), color=COLORS[col])

for tc, (t_lo, t_hi) in LN_WINDOWS.items():
    ax.axvspan((t_lo - t0)/60, (t_hi - t0)/60, alpha=0.18, color=COLORS[tc])

ax.axhline(LN2_BP_C, color='navy', lw=1.0, ls='--', alpha=0.7, label=f'LN₂ BP ({LN2_BP_C} °C)')
ax.set_xlabel('Time from log start [min]')
ax.set_ylabel('Raw temperature [°C]')
ax.set_title('Dip 1 — LN₂ immersion (shaded = plateau window)')
ax.legend(fontsize=9)
plt.show()

/tmp/ipykernel_1222632/775131639.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Dip detail — Dip 2 (THM, THI — partial immersion)

In [5]:
dip2_cols = ['THM_C', 'THI_C']
dip2_mask = (df['time_s'] >= 194_050) & (df['time_s'] <= 194_700)
dip2 = df.loc[dip2_mask].copy()

fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
for col in dip2_cols:
    mask = dip2[col].notna()
    ax.plot(dip2.loc[mask, 't_rel_min'], dip2.loc[mask, col],
            lw=1.2, label=col.replace('_C', ''), color=COLORS[col])

ax.axhline(LN2_BP_C, color='navy', lw=1.0, ls='--', alpha=0.7, label=f'LN₂ BP ({LN2_BP_C} °C)')
ax.set_xlabel('Time from log start [min]')
ax.set_ylabel('Raw temperature [°C]')
ax.set_title('Dip 2 — THM / THI partial immersion (did NOT reach LN₂ saturation)')
ax.legend(fontsize=9)
plt.show()

/tmp/ipykernel_1222632/3492772557.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Two-point calibration

For each TC with a valid LN plateau:
$$g = \frac{T_\text{LN} - T_\text{room}}{T_\text{raw,LN} - T_\text{raw,room}}, \quad b = T_\text{room} - g \cdot T_\text{raw,room}$$

`THM` and `THI` receive only a room-temperature single-point offset (gain forced to 1).

In [6]:
room_mask = (df['time_s'] >= ROOM_WINDOW[0]) & (df['time_s'] <= ROOM_WINDOW[1])

# Room-temperature raw anchors (median of pre-dip window)
room_raw = {col: float(df.loc[room_mask, col].median()) for col in TC_COLS if col in df.columns}

print('Room raw readings (pre-dip median):')
for col, val in room_raw.items():
    print(f'  {col:<10s}: {val:.3f} °C')

Room raw readings (pre-dip median):
  TTEST_C   : 20.050 °C
  TFO_C     : 20.050 °C
  TTI_C     : 19.990 °C
  TTO_C     : 18.430 °C
  TMI_C     : 19.290 °C
  THM_C     : 20.520 °C
  THI_C     : 20.480 °C


In [7]:
# LN plateau raw anchors
ln_raw = {}
for col, (t_lo, t_hi) in LN_WINDOWS.items():
    mask = (df['time_s'] >= t_lo) & (df['time_s'] <= t_hi)
    ln_raw[col] = float(df.loc[mask, col].median())

print('LN₂ plateau raw readings (stable window median):')
for col, val in ln_raw.items():
    print(f'  {col:<10s}: {val:.3f} °C  (offset from LN₂ BP: {val - LN2_BP_C:+.2f} °C)')

LN₂ plateau raw readings (stable window median):
  TTO_C     : -176.590 °C  (offset from LN₂ BP: +19.21 °C)
  TMI_C     : -175.460 °C  (offset from LN₂ BP: +20.34 °C)
  TFO_C     : -177.200 °C  (offset from LN₂ BP: +18.60 °C)
  TTI_C     : -176.855 °C  (offset from LN₂ BP: +18.95 °C)


In [8]:
records = []

for col in TC_COLS:
    if col not in room_raw or np.isnan(room_raw[col]):
        continue

    raw_room = room_raw[col]
    room_offset = ROOM_TRUE_C - raw_room

    if col in LN_WINDOWS:
        raw_ln = ln_raw[col]
        gain = (LN2_BP_C - ROOM_TRUE_C) / (raw_ln - raw_room)
        offset = ROOM_TRUE_C - gain * raw_room
        cal_type = 'two-point'
        # verification
        check_room = gain * raw_room + offset
        check_ln   = gain * raw_ln   + offset
    elif col in NO_LN_CHANNELS:
        gain   = 1.0
        offset = room_offset
        raw_ln = float('nan')
        cal_type = 'room-only'
        check_room = raw_room + offset
        check_ln   = float('nan')
    else:
        # TTEST_C — not dipped in this session, room offset only
        gain   = 1.0
        offset = room_offset
        raw_ln = float('nan')
        cal_type = 'room-only'
        check_room = raw_room + offset
        check_ln   = float('nan')

    records.append({
        'TC':            col.replace('_C', ''),
        'raw_room_C':    raw_room,
        'raw_ln_C':      raw_ln,
        'gain':          gain,
        'offset_C':      offset,
        'cal_type':      cal_type,
        'check_room_C':  check_room,
        'check_ln_C':    check_ln,
    })

cal = pd.DataFrame(records)

display_cols = ['TC', 'raw_room_C', 'raw_ln_C', 'gain', 'offset_C', 'cal_type']
print(cal[display_cols].to_string(index=False, float_format='{:.4f}'.format))

   TC  raw_room_C  raw_ln_C   gain  offset_C  cal_type
TTEST     20.0500       NaN 1.0000    0.2278 room-only
  TFO     20.0500 -177.2000 1.0955   -1.6860 two-point
  TTI     19.9900 -176.8550 1.0977   -1.6653 two-point
  TTO     18.4300 -176.5900 1.1080   -0.1422 two-point
  TMI     19.2900 -175.4600 1.1095   -1.1247 two-point
  THM     20.5200       NaN 1.0000   -0.2422 room-only
  THI     20.4800       NaN 1.0000   -0.2022 room-only


## Calibration curves

In [9]:
fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)

raw_grid = np.linspace(LN2_BP_C - 5, ROOM_TRUE_C + 5, 300)

for _, row in cal.iterrows():
    tc   = row['TC']
    col  = tc + '_C'
    cal_line = row['gain'] * raw_grid + row['offset_C']
    ls = '--' if row['cal_type'] == 'room-only' else '-'
    ax.plot(raw_grid, cal_line, lw=1.8, ls=ls, color=COLORS.get(col, 'grey'), label=tc)

    # room anchor
    ax.scatter(row['raw_room_C'], ROOM_TRUE_C, color=COLORS.get(col, 'grey'), marker='o', s=50, zorder=4)
    # LN anchor
    if not np.isnan(row['raw_ln_C']):
        ax.scatter(row['raw_ln_C'], LN2_BP_C, color=COLORS.get(col, 'grey'), marker='s', s=50, zorder=4)

ax.axline((0, 0), slope=1.0, color='0.7', lw=0.8, ls=':', label='unity (no correction)')
ax.scatter([], [], marker='o', color='k', s=40, label='Room anchor')
ax.scatter([], [], marker='s', color='k', s=40, label='LN₂ anchor')
ax.axhline(LN2_BP_C, color='navy', lw=0.8, ls='--', alpha=0.5)
ax.axhline(ROOM_TRUE_C, color='green', lw=0.8, ls='--', alpha=0.5)

ax.set_xlabel('Raw temperature [°C]')
ax.set_ylabel('Calibrated temperature [°C]')
ax.set_title('Two-point calibration curves (solid) and room-only offsets (dashed)')
ax.legend(fontsize=8, ncol=2)
plt.show()

/tmp/ipykernel_1222632/3065090712.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Calibration summary table

In [10]:
from IPython.display import display

summary = cal[['TC', 'gain', 'offset_C', 'cal_type', 'raw_room_C', 'check_room_C', 'raw_ln_C', 'check_ln_C']].copy()
summary['affine_law'] = summary.apply(
    lambda r: f"T_cal = {r['gain']:.5f} × T_raw + ({r['offset_C']:+.3f})", axis=1
)
summary['room_residual_C'] = summary['check_room_C'] - ROOM_TRUE_C
summary['ln_residual_C']   = summary['check_ln_C'] - LN2_BP_C

out = summary[['TC', 'affine_law', 'cal_type', 'room_residual_C', 'ln_residual_C']]
display(out.style.format({'room_residual_C': '{:.4f}', 'ln_residual_C': '{:.4f}'}).hide(axis='index'))

TC,affine_law,cal_type,room_residual_C,ln_residual_C
TTEST,T_cal = 1.00000 × T_raw + (+0.228),room-only,0.0000,nan
TFO,T_cal = 1.09545 × T_raw + (-1.686),two-point,0.0000,0.0000
TTI,T_cal = 1.09771 × T_raw + (-1.665),two-point,0.0000,0.0000
TTO,T_cal = 1.10798 × T_raw + (-0.142),two-point,0.0000,0.0000
TMI,T_cal = 1.10951 × T_raw + (-1.125),two-point,0.0000,-0.0000
THM,T_cal = 1.00000 × T_raw + (-0.242),room-only,0.0000,nan
THI,T_cal = 1.00000 × T_raw + (-0.202),room-only,0.0000,nan


## Apply calibration and verify

In [11]:
df_cal = df.copy()
for _, row in cal.iterrows():
    col     = row['TC'] + '_C'
    col_cal = row['TC'] + '_cal_C'
    df_cal[col_cal] = row['gain'] * df_cal[col] + row['offset_C']

cal_cols = [row['TC'] + '_cal_C' for _, row in cal.iterrows()]

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True, constrained_layout=True)

# raw
ax = axes[0]
for col in TC_COLS:
    mask = df_cal[col].notna()
    ax.plot(df_cal.loc[mask, 't_rel_min'], df_cal.loc[mask, col],
            lw=0.8, alpha=0.7, color=COLORS[col], label=col.replace('_C', ''))
ax.axhline(LN2_BP_C, color='navy', lw=0.8, ls='--', alpha=0.5)
ax.axhline(ROOM_TRUE_C, color='green', lw=0.8, ls='--', alpha=0.5)
ax.set_ylabel('Raw [°C]')
ax.set_title('Raw')
ax.legend(fontsize=7, ncol=4)

# calibrated
ax = axes[1]
for _, row in cal.iterrows():
    col_cal = row['TC'] + '_cal_C'
    col_raw = row['TC'] + '_C'
    mask    = df_cal[col_cal].notna()
    ls = '--' if row['cal_type'] == 'room-only' else '-'
    ax.plot(df_cal.loc[mask, 't_rel_min'], df_cal.loc[mask, col_cal],
            lw=0.9, ls=ls, color=COLORS[col_raw], label=row['TC'])
ax.axhline(LN2_BP_C, color='navy', lw=0.8, ls='--', alpha=0.5, label=f'LN₂ BP ({LN2_BP_C} °C)')
ax.axhline(ROOM_TRUE_C, color='green', lw=0.8, ls='--', alpha=0.5, label=f'Room ({ROOM_TRUE_F:.1f} °F / {ROOM_TRUE_C:.3f} °C)')
ax.set_xlabel('Time from log start [min]')
ax.set_ylabel('Calibrated [°C]')
ax.set_title('Calibrated (solid = two-point, dashed = room-only)')
ax.legend(fontsize=7, ncol=4)

plt.show()

/tmp/ipykernel_1222632/1566742909.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Export calibration coefficients

In [12]:
export = cal[['TC', 'gain', 'offset_C', 'cal_type']].copy()
export_path = repo_root / 'data' / 'processed' / 'calibration' / 'TC_calibration_20260410.csv'
export_path.parent.mkdir(parents=True, exist_ok=True)
export.to_csv(export_path, index=False)
print(f'Saved → {export_path}')
display(export)

Saved → /home/aamy/Documents/hfe-system/data/processed/calibration/TC_calibration_20260410.csv


,TC,gain,offset_C,cal_type
0,TTEST,1.000000,0.227778,room-only
1,TFO,1.095451,-1.686022,two-point
2,TTI,1.097705,-1.665349,two-point
3,TTO,1.107978,-0.142248,two-point
4,TMI,1.109514,-1.124740,two-point
5,THM,1.000000,-0.242222,room-only
6,THI,1.000000,-0.202222,room-only


## April 20, 2026 recalibration — corrected thermocouple types

**Source file:** `log_20260420_111545.csv`

This second calibration run was recorded after fixing the firmware so the installed `T`-type probes are no longer decoded as `K`-type.

- The room anchor uses the flat tail of the log (last 5 minutes) with the measured ambient `68.5 °F = 20.278 °C`.
- `TTEST`, `TFO`, `TMI`, and `TTO` each show a clear LN plateau in this file.
- `TTI` was dipped twice: once with the original probe and then with the longer replacement probe intended for future runs. The exported `TTI` coefficient below uses the **longer** probe.
- In the recorded data, `THM` and `THI` still do **not** show an LN plateau, so this log can only provide room-anchor offsets for those channels.


In [13]:
CAL_LOG_NEW = repo_root / 'data' / 'raw' / 'calibration' / 'log_20260420_111545.csv'

ROOM_TRUE_F_NEW = 68.5
ROOM_TRUE_C_NEW = (ROOM_TRUE_F_NEW - 32.0) * 5.0 / 9.0
ROOM_WINDOW_DURATION_S_NEW = 300.0  # fit room anchor from the flat end-of-log segment

TC_COLS_NEW = ['TTEST_C', 'TFO_C', 'TTI_C', 'TTO_C', 'TMI_C', 'THI_C', 'THM_C']
COLORS_NEW = dict(zip(TC_COLS_NEW, ['C0', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6']))

LN_WINDOWS_NEW = {
    'TTEST_C': (173.366, 233.590),
    'TFO_C':   (1193.035, 1286.489),
    'TMI_C':   (1805.669, 1886.661),
    'TTO_C':   (2586.518, 2661.280),
}
TTI_WINDOWS_NEW = {
    'TTI_original': (3290.528, 3371.522),
    'TTI_long':     (3587.502, 3668.493),
}
EXPORT_LN_WINDOWS_NEW = {
    **LN_WINDOWS_NEW,
    'TTI_C': TTI_WINDOWS_NEW['TTI_long'],
}
ROOM_ONLY_NEW = ['THI_C', 'THM_C']


import orca

# Normalize the 10-channel block before using THI/THM names: U8 is THI_C and U9 is THM_C.
df_new = orca.canonicalize_tc_columns(pd.read_csv(CAL_LOG_NEW))
df_new['time_s'] = pd.to_numeric(df_new['time_s'], errors='coerce')
for col in [*TC_COLS_NEW, 'fluid_temperature_c']:
    if col in df_new.columns:
        df_new[col] = pd.to_numeric(df_new[col], errors='coerce')

df_new = df_new.dropna(subset=['time_s']).sort_values('time_s').reset_index(drop=True)
df_new['t_rel_min'] = (df_new['time_s'] - df_new['time_s'].iloc[0]) / 60.0

room_t_hi_new = float(df_new['time_s'].iloc[-1])
room_t_lo_new = room_t_hi_new - ROOM_WINDOW_DURATION_S_NEW
ROOM_WINDOW_NEW = (room_t_lo_new, room_t_hi_new)

print(f'Rows: {len(df_new)}')
print(f"Time range: {df_new['time_s'].iloc[0]:.3f} – {df_new['time_s'].iloc[-1]:.3f} s")
print(f"Duration: {df_new['t_rel_min'].iloc[-1]:.1f} min")
print(f"Room window (flat tail): {ROOM_WINDOW_NEW[0]:.3f} – {ROOM_WINDOW_NEW[1]:.3f} s")
print(f"Room reference: {ROOM_TRUE_F_NEW:.1f} °F / {ROOM_TRUE_C_NEW:.3f} °C")


Rows: 2734
Time range: 173.366 – 5849.052 s
Duration: 94.6 min
Room window (flat tail): 5549.052 – 5849.052 s
Room reference: 68.5 °F / 20.278 °C


In [14]:
fig, ax = plt.subplots(figsize=(12, 5.5), constrained_layout=True)

for col in TC_COLS_NEW:
    mask = df_new[col].notna()
    ax.plot(
        df_new.loc[mask, 't_rel_min'],
        df_new.loc[mask, col],
        lw=1.0,
        color=COLORS_NEW[col],
        label=col.replace('_C', ''),
    )

ax.axvspan(
    (ROOM_WINDOW_NEW[0] - df_new['time_s'].iloc[0]) / 60.0,
    (ROOM_WINDOW_NEW[1] - df_new['time_s'].iloc[0]) / 60.0,
    alpha=0.12,
    color='green',
    label='Room anchor (flat end segment)',
)
for tc, (t_lo, t_hi) in LN_WINDOWS_NEW.items():
    ax.axvspan(
        (t_lo - df_new['time_s'].iloc[0]) / 60.0,
        (t_hi - df_new['time_s'].iloc[0]) / 60.0,
        alpha=0.12,
        color=COLORS_NEW[tc],
    )
for label, (t_lo, t_hi) in TTI_WINDOWS_NEW.items():
    ax.axvspan(
        (t_lo - df_new['time_s'].iloc[0]) / 60.0,
        (t_hi - df_new['time_s'].iloc[0]) / 60.0,
        alpha=0.12,
        color='tab:purple' if label == 'TTI_long' else 'tab:orange',
    )

ax.axhline(LN2_BP_C, color='navy', lw=1.0, ls='--', alpha=0.7, label=f'LN₂ boiling point ({LN2_BP_C} °C)')
ax.axhline(ROOM_TRUE_C_NEW, color='green', lw=1.0, ls='--', alpha=0.7, label=f'Room reference ({ROOM_TRUE_F_NEW:.1f} °F / {ROOM_TRUE_C_NEW:.3f} °C)')
ax.set_xlabel('Time from log start [min]')
ax.set_ylabel('Raw temperature [°C]')
ax.set_title('TC calibration session — log_20260420_111545.csv')
ax.legend(loc='lower right', fontsize=8, ncol=2)
plt.show()


/tmp/ipykernel_1222632/946802268.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
room_mask_new = df_new['time_s'].between(*ROOM_WINDOW_NEW)
room_raw_new = {
    col: float(df_new.loc[room_mask_new, col].median())
    for col in TC_COLS_NEW
    if col in df_new.columns
}
flow_room_new = float(df_new.loc[room_mask_new, 'fluid_temperature_c'].median()) if 'fluid_temperature_c' in df_new.columns else float('nan')

print('Room raw readings (flat tail median):')
for col, val in room_raw_new.items():
    print(f'  {col:<10s}: {val:.3f} °C')
print(f'  flow meter : {flow_room_new:.3f} °C')

probe_rows_new = []
for tc, window in LN_WINDOWS_NEW.items():
    mask = df_new['time_s'].between(*window)
    raw_ln = float(df_new.loc[mask, tc].median())
    raw_room = room_raw_new[tc]
    gain = (LN2_BP_C - ROOM_TRUE_C_NEW) / (raw_ln - raw_room)
    offset = ROOM_TRUE_C_NEW - gain * raw_room
    probe_rows_new.append({
        'probe': tc.replace('_C', ''),
        'column': tc,
        'window_start_s': window[0],
        'window_end_s': window[1],
        'raw_room_C': raw_room,
        'raw_ln_C': raw_ln,
        'gain': gain,
        'offset_C': offset,
        'note': 'single production probe',
    })

for label, window in TTI_WINDOWS_NEW.items():
    mask = df_new['time_s'].between(*window)
    raw_ln = float(df_new.loc[mask, 'TTI_C'].median())
    raw_room = room_raw_new['TTI_C']
    gain = (LN2_BP_C - ROOM_TRUE_C_NEW) / (raw_ln - raw_room)
    offset = ROOM_TRUE_C_NEW - gain * raw_room
    probe_rows_new.append({
        'probe': 'TTI',
        'column': 'TTI_C',
        'window_start_s': window[0],
        'window_end_s': window[1],
        'raw_room_C': raw_room,
        'raw_ln_C': raw_ln,
        'gain': gain,
        'offset_C': offset,
        'note': 'original probe' if label == 'TTI_original' else 'long probe (future runs)',
    })

probe_summary_new = pd.DataFrame(probe_rows_new)
display(
    probe_summary_new[['probe', 'note', 'raw_room_C', 'raw_ln_C', 'gain', 'offset_C']]
    .style.format({'raw_room_C': '{:.3f}', 'raw_ln_C': '{:.3f}', 'gain': '{:.5f}', 'offset_C': '{:+.4f}'})
    .hide(axis='index')
)


Room raw readings (flat tail median):
  TTEST_C   : 20.330 °C
  TFO_C     : 20.870 °C
  TTI_C     : 21.230 °C
  TTO_C     : 20.340 °C
  TMI_C     : 20.400 °C
  THI_C     : 20.920 °C
  THM_C     : 19.690 °C
  flow meter : 20.713 °C


probe,note,raw_room_C,raw_ln_C,gain,offset_C
TTEST,single production probe,20.330,-192.265,1.01638,-0.3853
TFO,single production probe,20.870,-194.860,1.00161,-0.6259
TMI,single production probe,20.400,-194.905,1.00359,-0.1954
TTO,single production probe,20.340,-194.410,1.00618,-0.1880
TTI,original probe,21.230,-195.240,0.99819,-0.9138
TTI,long probe (future runs),21.230,-195.465,0.99715,-0.8918


In [16]:
records_new = []

for col in TC_COLS_NEW:
    raw_room = room_raw_new[col]
    room_offset = ROOM_TRUE_C_NEW - raw_room

    if col in EXPORT_LN_WINDOWS_NEW:
        t_lo, t_hi = EXPORT_LN_WINDOWS_NEW[col]
        raw_ln = float(df_new.loc[df_new['time_s'].between(t_lo, t_hi), col].median())
        gain = (LN2_BP_C - ROOM_TRUE_C_NEW) / (raw_ln - raw_room)
        offset = ROOM_TRUE_C_NEW - gain * raw_room
        cal_type = 'two-point'
        note = 'TTI long probe' if col == 'TTI_C' else 'April 20 LN plateau'
    elif col in ROOM_ONLY_NEW:
        gain = 1.0
        offset = room_offset
        raw_ln = float('nan')
        cal_type = 'room-only'
        note = 'No LN plateau visible in April 20 log'
    else:
        gain = 1.0
        offset = room_offset
        raw_ln = float('nan')
        cal_type = 'room-only'
        note = 'No LN plateau assigned'

    records_new.append({
        'TC': col.replace('_C', ''),
        'raw_room_C': raw_room,
        'raw_ln_C': raw_ln,
        'gain': gain,
        'offset_C': offset,
        'cal_type': cal_type,
        'note': note,
    })

cal_new = pd.DataFrame(records_new)

display(
    cal_new[['TC', 'gain', 'offset_C', 'cal_type', 'raw_room_C', 'raw_ln_C', 'note']]
    .style.format({'gain': '{:.5f}', 'offset_C': '{:+.4f}', 'raw_room_C': '{:.3f}', 'raw_ln_C': '{:.3f}'})
    .hide(axis='index')
)

export_new = cal_new[['TC', 'gain', 'offset_C', 'cal_type']].copy()
print('Preliminary April 20 table with room-only HX rows; final export is written by the warmup-transfer section below.')
display(export_new)


TC,gain,offset_C,cal_type,raw_room_C,raw_ln_C,note
TTEST,1.01638,-0.3853,two-point,20.330,-192.265,April 20 LN plateau
TFO,1.00161,-0.6259,two-point,20.870,-194.860,April 20 LN plateau
TTI,0.99715,-0.8918,two-point,21.230,-195.465,TTI long probe
TTO,1.00618,-0.1880,two-point,20.340,-194.410,April 20 LN plateau
TMI,1.00359,-0.1954,two-point,20.400,-194.905,April 20 LN plateau
THI,1.00000,-0.6422,room-only,20.920,nan,No LN plateau visible in April 20 log
THM,1.00000,+0.5878,room-only,19.690,nan,No LN plateau visible in April 20 log


Preliminary April 20 table with room-only HX rows; final export is written by the warmup-transfer section below.


,TC,gain,offset_C,cal_type
0,TTEST,1.016382,-0.385273,two-point
1,TFO,1.001612,-0.625867,two-point
2,TTI,0.997152,-0.891752,two-point
3,TTO,1.006183,-0.187982,two-point
4,TMI,1.003589,-0.195442,two-point
5,THI,1.000000,-0.642222,room-only
6,THM,1.000000,0.587778,room-only


In [17]:
fig, ax = plt.subplots(figsize=(8.5, 6.2), constrained_layout=True)

raw_grid_new = np.linspace(LN2_BP_C - 5.0, ROOM_TRUE_C_NEW + 5.0, 400)

for _, row in cal_new.iterrows():
    tc = row['TC']
    col = f'{tc}_C'
    label = 'TTI (long probe)' if row['note'] == 'TTI long probe' else tc
    ls = '--' if row['cal_type'] == 'room-only' else '-'
    ax.plot(
        raw_grid_new,
        row['gain'] * raw_grid_new + row['offset_C'],
        lw=1.8,
        ls=ls,
        color=COLORS_NEW.get(col, '0.4'),
        label=label,
    )
    ax.scatter(row['raw_room_C'], ROOM_TRUE_C_NEW, color=COLORS_NEW.get(col, '0.4'), marker='o', s=50, zorder=4)
    if np.isfinite(row['raw_ln_C']):
        ax.scatter(row['raw_ln_C'], LN2_BP_C, color=COLORS_NEW.get(col, '0.4'), marker='s', s=50, zorder=4)

tti_original = probe_summary_new[probe_summary_new['note'] == 'original probe'].iloc[0]
ax.plot(
    raw_grid_new,
    tti_original['gain'] * raw_grid_new + tti_original['offset_C'],
    lw=1.8,
    ls='-.',
    color=COLORS_NEW['TTI_C'],
    alpha=0.95,
    label='TTI (original probe)',
)
ax.scatter(tti_original['raw_room_C'], ROOM_TRUE_C_NEW, color=COLORS_NEW['TTI_C'], marker='^', s=55, zorder=5)
ax.scatter(tti_original['raw_ln_C'], LN2_BP_C, color=COLORS_NEW['TTI_C'], marker='D', s=50, zorder=5)

ax.axline((0, 0), slope=1.0, color='0.70', lw=0.9, ls=':', label='unity (no correction)')
ax.scatter([], [], marker='o', color='k', s=40, label='Room anchor')
ax.scatter([], [], marker='s', color='k', s=40, label='LN anchor')
ax.scatter([], [], marker='^', color='k', s=40, label='TTI original room anchor')
ax.scatter([], [], marker='D', color='k', s=40, label='TTI original LN anchor')
ax.axhline(LN2_BP_C, color='navy', lw=0.8, ls='--', alpha=0.5)
ax.axhline(ROOM_TRUE_C_NEW, color='green', lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel('Raw temperature [°C]')
ax.set_ylabel('Calibrated temperature [°C]')
ax.set_title('April 20 calibration curves (future TTI = long probe)')
ax.legend(fontsize=8, ncol=2)
plt.show()


/tmp/ipykernel_1222632/79167567.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## April 24 warmup-transfer calibration for installed HX channels

This final export keeps the April 20 room and LN anchors for `TTEST`, `TFO`, `TTI`, `TTO`, and `TMI`, then replaces the installed HX channel rows with a room + warmup-transfer fit. The warmup anchor is taken from the latest recirculation log after cooldown stops: final cooldown-valve close + 5 minutes through pump-off, filtered to the stable band where old-calibrated `TTI` is between `-35.5 C` and `-34.5 C`.

The calibration remains channel-first: logger channel `U8`/`THI_C` and logger channel `U9`/`THM_C` get their own coefficients, without resolving the physical THI/THM label swap. The April 20 calibration log had historical `THM_C, THI_C` header names in those two slots, so the room anchor is normalized by hardware position before fitting.


In [18]:
from IPython.display import Markdown, display
import orca

WARMUP_TRANSFER_CURRENT_LOG = repo_root / 'data' / 'raw' / 'recirculation' / 'log_20260424_153546.csv'
WARMUP_TRANSFER_ARCHIVED_LOG = (
    repo_root
    / 'data'
    / 'raw'
    / 'archive'
    / 'tc_calibrated_20260420_originals'
    / 'recirculation'
    / 'log_20260424_153546.csv'
)
WARMUP_TRANSFER_LOG = WARMUP_TRANSFER_ARCHIVED_LOG if WARMUP_TRANSFER_ARCHIVED_LOG.exists() else WARMUP_TRANSFER_CURRENT_LOG
WARMUP_TRANSFER_DELAY_MIN = 5.0
WARMUP_TRANSFER_TTI_BAND_C = (-35.5, -34.5)
WARMUP_TRANSFER_TTI_BAND_TOL_C = 1e-4
WARMUP_TRANSFER_LOOP_COLS = ['TFO_C', 'TTI_C', 'TTO_C', 'TMI_C']
WARMUP_TRANSFER_HX_COLS = ['THI_C', 'THM_C']

APR20_PREVIOUS_CALIBRATION = pd.DataFrame([
    {'TC': 'TTEST', 'gain': 1.0163822186682556, 'offset_C': -0.38527272774785715, 'cal_type': 'two-point'},
    {'TC': 'TFO', 'gain': 1.0016120974263096, 'offset_C': -0.6258666955093055, 'cal_type': 'two-point'},
    {'TC': 'TTI', 'gain': 0.9971516545272286, 'offset_C': -0.8917518478352875, 'cal_type': 'two-point'},
    {'TC': 'TTO', 'gain': 1.006182900012935, 'offset_C': -0.18798240848531833, 'cal_type': 'two-point'},
    {'TC': 'TMI', 'gain': 1.0035892235562471, 'offset_C': -0.19544238276966297, 'cal_type': 'two-point'},
    {'TC': 'THI', 'gain': 1.0, 'offset_C': 0.5877777777777773, 'cal_type': 'room-only'},
    {'TC': 'THM', 'gain': 1.0, 'offset_C': -0.6422222222222231, 'cal_type': 'room-only'},
])


def _first_comment_line(path: Path) -> str:
    with path.open('r', encoding='utf-8', newline='') as handle:
        first = handle.readline().strip()
    return first if first.startswith('#') else ''


def _invert_calibrated_log_to_raw(frame: pd.DataFrame, calibration: pd.DataFrame) -> pd.DataFrame:
    raw = frame.copy()
    for row in calibration.to_dict('records'):
        column = f"{row['TC']}_C"
        if column not in raw.columns:
            continue
        values = pd.to_numeric(raw[column], errors='coerce')
        raw[column] = (values - float(row['offset_C'])) / float(row['gain'])
    return raw


warmup_comment = _first_comment_line(WARMUP_TRANSFER_LOG)
warmup_logged = orca.canonicalize_tc_columns(pd.read_csv(WARMUP_TRANSFER_LOG, comment='#'))
if 'tc_calibrated=true' in warmup_comment.lower():
    warmup_raw = _invert_calibrated_log_to_raw(warmup_logged, APR20_PREVIOUS_CALIBRATION)
    warmup_input_note = 'source log was April-20 calibrated; inverted to raw before fitting'
else:
    warmup_raw = warmup_logged.copy()
    warmup_input_note = 'source log is raw/restored; using TC columns directly before fitting'

warmup_oldcal = orca.apply_tc_calibration(warmup_raw, APR20_PREVIOUS_CALIBRATION, preserve_raw=False)
for column in ['time_s', 'valve', 'pump_cmd_pct', *WARMUP_TRANSFER_LOOP_COLS, *WARMUP_TRANSFER_HX_COLS]:
    if column in warmup_oldcal.columns:
        warmup_oldcal[column] = pd.to_numeric(warmup_oldcal[column], errors='coerce')
    if column in warmup_raw.columns:
        warmup_raw[column] = pd.to_numeric(warmup_raw[column], errors='coerce')

valid_time = warmup_oldcal['time_s'].notna()
warmup_oldcal = warmup_oldcal.loc[valid_time].copy()
warmup_raw = warmup_raw.loc[valid_time].copy()
sort_order = warmup_oldcal.sort_values('time_s').index
warmup_oldcal = warmup_oldcal.loc[sort_order].reset_index(drop=True)
warmup_raw = warmup_raw.loc[sort_order].reset_index(drop=True)
warmup_oldcal['elapsed_from_log_start_min'] = (warmup_oldcal['time_s'] - float(warmup_oldcal['time_s'].iloc[0])) / 60.0
warmup_raw['elapsed_from_log_start_min'] = warmup_oldcal['elapsed_from_log_start_min']

valve_state = warmup_oldcal['valve'].fillna(0.0)
pump_cmd = warmup_oldcal['pump_cmd_pct'].fillna(0.0)
pump_off_candidates = warmup_oldcal.loc[
    pump_cmd.le(0.0) & pump_cmd.shift(fill_value=float(pump_cmd.iloc[0])).gt(0.0),
    'elapsed_from_log_start_min',
]
if pump_off_candidates.empty:
    raise RuntimeError('Could not find pump-off command transition in warmup-transfer log.')
pump_off_min = float(pump_off_candidates.iloc[0])

valve_falling_edges = warmup_oldcal.loc[
    valve_state.le(0.0) & valve_state.shift(fill_value=float(valve_state.iloc[0])).gt(0.0),
    'elapsed_from_log_start_min',
]
valve_falling_edges = valve_falling_edges[valve_falling_edges.lt(pump_off_min)]
if valve_falling_edges.empty:
    raise RuntimeError('Could not find final cooldown-valve close before pump-off.')
cooldown_stopped_min = float(valve_falling_edges.iloc[-1])
warmup_start_min = cooldown_stopped_min + WARMUP_TRANSFER_DELAY_MIN

warmup_mask = warmup_oldcal['elapsed_from_log_start_min'].ge(warmup_start_min) & warmup_oldcal['elapsed_from_log_start_min'].lt(pump_off_min)
warmup_oldcal_window = warmup_oldcal.loc[warmup_mask].copy()
warmup_raw_window = warmup_raw.loc[warmup_mask].copy()

band_mask = warmup_oldcal_window['TTI_C'].between(
    WARMUP_TRANSFER_TTI_BAND_C[0] - WARMUP_TRANSFER_TTI_BAND_TOL_C,
    WARMUP_TRANSFER_TTI_BAND_C[1] + WARMUP_TRANSFER_TTI_BAND_TOL_C,
)
warmup_oldcal_band = warmup_oldcal_window.loc[band_mask].copy()
warmup_raw_band = warmup_raw_window.loc[band_mask].copy()
if len(warmup_oldcal_band) < 20:
    raise RuntimeError(f'Warmup-transfer band has too few samples: {len(warmup_oldcal_band)}')

loop_reference_series = warmup_oldcal_band[WARMUP_TRANSFER_LOOP_COLS].mean(axis=1)
warmup_reference_c = float(loop_reference_series.median())
loop_span_c = warmup_oldcal_band[WARMUP_TRANSFER_LOOP_COLS].max(axis=1) - warmup_oldcal_band[WARMUP_TRANSFER_LOOP_COLS].min(axis=1)

warmup_transfer_rows = []
for column in WARMUP_TRANSFER_HX_COLS:
    tag = column.replace('_C', '')
    raw_room = float(room_raw_new[column])
    raw_warmup = float(warmup_raw_band[column].median())
    gain = (warmup_reference_c - ROOM_TRUE_C_NEW) / (raw_warmup - raw_room)
    offset = ROOM_TRUE_C_NEW - gain * raw_room
    warmup_transfer_rows.append({
        'TC': tag,
        'column': column,
        'raw_room_C': raw_room,
        'raw_warmup_C': raw_warmup,
        'warmup_reference_C': warmup_reference_c,
        'gain': gain,
        'offset_C': offset,
        'cal_type': 'room+warmup-transfer',
        'room_check_C': gain * raw_room + offset,
        'warmup_check_C': gain * raw_warmup + offset,
    })

warmup_transfer_summary = pd.DataFrame(warmup_transfer_rows)
display(Markdown(
    f"Warmup-transfer source: `{WARMUP_TRANSFER_LOG.relative_to(repo_root)}`; `{warmup_input_note}`. "
    f"Cooldown stopped at `{cooldown_stopped_min:.2f} min`; fit window starts at `{warmup_start_min:.2f} min`; "
    f"pump-off at `{pump_off_min:.2f} min`."
))

display(pd.DataFrame([{
    'band_samples': int(len(warmup_oldcal_band)),
    'TTI_band_low_C': WARMUP_TRANSFER_TTI_BAND_C[0],
    'TTI_band_high_C': WARMUP_TRANSFER_TTI_BAND_C[1],
    'loop_reference_median_C': warmup_reference_c,
    'loop_span_median_C': float(loop_span_c.median()),
    'TTI_median_C': float(warmup_oldcal_band['TTI_C'].median()),
    'TTO_median_C': float(warmup_oldcal_band['TTO_C'].median()),
}]).style.format({
    'band_samples': '{:.0f}',
    'TTI_band_low_C': '{:.1f}',
    'TTI_band_high_C': '{:.1f}',
    'loop_reference_median_C': '{:.4f}',
    'loop_span_median_C': '{:.4f}',
    'TTI_median_C': '{:.4f}',
    'TTO_median_C': '{:.4f}',
}).hide(axis='index'))

display(warmup_transfer_summary.style.format({
    'raw_room_C': '{:.4f}',
    'raw_warmup_C': '{:.4f}',
    'warmup_reference_C': '{:.4f}',
    'gain': '{:.12f}',
    'offset_C': '{:+.12f}',
    'room_check_C': '{:.4f}',
    'warmup_check_C': '{:.4f}',
}).hide(axis='index'))


mixed_label_rejected = pd.DataFrame([
    {
        'fit': 'rejected mixed-label',
        'TC': 'THI',
        'problem': 'U8 warmup was paired with historical U9 room label',
        'gain': 1.0125805664558891,
        'offset_C': 0.3400664242613196,
    },
    {
        'fit': 'rejected mixed-label',
        'TC': 'THM',
        'problem': 'U9 warmup was paired with historical U8 room label',
        'gain': 1.0429249036689563,
        'offset_C': -1.5402112069767888,
    },
])
accepted_channel_fit = warmup_transfer_summary.assign(
    fit='accepted channel-consistent',
    problem='U8 room with U8 warmup, U9 room with U9 warmup',
)[['fit', 'TC', 'problem', 'gain', 'offset_C']]
channel_fit_comparison = pd.concat(
    [accepted_channel_fit, mixed_label_rejected],
    ignore_index=True,
)
display(Markdown(
    'The earlier mixed-label fit is rejected because it crossed hardware channels between the '
    'April 20 room log and the April 24 warmup log. The exported coefficients below use U8 with U8 '
    'and U9 with U9.'
))
display(channel_fit_comparison.style.format({
    'gain': '{:.12f}',
    'offset_C': '{:+.12f}',
}).hide(axis='index'))

final_calibration = cal_new[['TC', 'gain', 'offset_C', 'cal_type']].copy()
for _, row in warmup_transfer_summary.iterrows():
    mask = final_calibration['TC'].eq(row['TC'])
    final_calibration.loc[mask, 'gain'] = row['gain']
    final_calibration.loc[mask, 'offset_C'] = row['offset_C']
    final_calibration.loc[mask, 'cal_type'] = row['cal_type']

final_order = ['TTEST', 'TFO', 'TTI', 'TTO', 'TMI', 'THI', 'THM']
final_calibration['TC'] = pd.Categorical(final_calibration['TC'], categories=final_order, ordered=True)
final_calibration = final_calibration.sort_values('TC').reset_index(drop=True)
final_calibration['TC'] = final_calibration['TC'].astype(str)

export_path_new = repo_root / 'data' / 'processed' / 'calibration' / 'TC_calibration_20260420.csv'
export_path_new.parent.mkdir(parents=True, exist_ok=True)
final_calibration.to_csv(export_path_new, index=False)
print(f'Saved final room + warmup-transfer calibration -> {export_path_new}')
display(final_calibration)


Warmup-transfer source: `data/raw/archive/tc_calibrated_20260420_originals/recirculation/log_20260424_153546.csv`; `source log was April-20 calibrated; inverted to raw before fitting`. Cooldown stopped at `174.13 min`; fit window starts at `179.13 min`; pump-off at `273.08 min`.

band_samples,TTI_band_low_C,TTI_band_high_C,loop_reference_median_C,loop_span_median_C,TTI_median_C,TTO_median_C
79,-35.5,-34.5,-35.0575,0.3900,-34.9900,-35.0800


TC,column,raw_room_C,raw_warmup_C,warmup_reference_C,gain,offset_C,cal_type,room_check_C,warmup_check_C
THI,THI_C,20.9200,-34.9578,-35.0575,0.990291310400,-0.439116435784,room+warmup-transfer,20.2778,-35.0575
THM,THM_C,19.6900,-32.1378,-35.0575,1.067676063887,-0.744763920153,room+warmup-transfer,20.2778,-35.0575


The earlier mixed-label fit is rejected because it crossed hardware channels between the April 20 room log and the April 24 warmup log. The exported coefficients below use U8 with U8 and U9 with U9.

fit,TC,problem,gain,offset_C
accepted channel-consistent,THI,"U8 room with U8 warmup, U9 room with U9 warmup",0.990291310400,-0.439116435784
accepted channel-consistent,THM,"U8 room with U8 warmup, U9 room with U9 warmup",1.067676063887,-0.744763920153
rejected mixed-label,THI,U8 warmup was paired with historical U9 room label,1.012580566456,+0.340066424261
rejected mixed-label,THM,U9 warmup was paired with historical U8 room label,1.042924903669,-1.540211206977


Saved final room + warmup-transfer calibration -> /home/aamy/Documents/hfe-system/data/processed/calibration/TC_calibration_20260420.csv


,TC,gain,offset_C,cal_type
0,TTEST,1.016382,-0.385273,two-point
1,TFO,1.001612,-0.625867,two-point
2,TTI,0.997152,-0.891752,two-point
3,TTO,1.006183,-0.187982,two-point
4,TMI,1.003589,-0.195442,two-point
5,THI,0.990291,-0.439116,room+warmup-transfer
6,THM,1.067676,-0.744764,room+warmup-transfer


### Warmup-transfer interpretation

The warmup-transfer point is not a bath-style absolute anchor. It is an in-situ transfer anchor taken when the loop probes are well mixed and `TTI` and `TTO` agree closely. The resulting HX coefficients should be used for UI display and in-memory notebook analysis, while raw logger CSV files remain uncalibrated.


## Legacy note — older runs with the wrong-type firmware

The April 20 coefficients above are the right reference for **future runs recorded with the corrected per-channel thermocouple types**, using the longer `TTI` probe.

They should **not** be applied directly to older logs recorded while the `T`-type probes were still being decoded as `K`-type in firmware. Those historical temperatures were linearised through the wrong thermocouple table, so recovering them requires a separate remapping from the stored wrong-type temperatures (or the equivalent thermocouple EMF) before any final per-probe offset/gain correction is applied.

The shared flow-log review helpers now do that historical remapping automatically for pre-fix logs: they back-convert the logged wrong-type temperatures to estimated `T`-type values with an effective cold-junction temperature fitted from the old April 10 and new April 20 calibration runs, then room-anchor the loop probes on the warmest stable flow-meter segment in that run.


## May 6, 2026 installed HX TC response in nitrogen gas

**Source file:** `data/raw/log_20260506_190814.csv`

This log was recorded with the tank filled with gaseous nitrogen and the LN valve opened. It is useful as an **installed-response diagnostic** for the two heat-exchanger thermocouples, not as an absolute calibration anchor.

Geometry notes for interpretation:

- `THI` is the heat-exchanger inlet / top-of-coil thermocouple.
- `THM` is intended to be the middle-of-coil thermocouple, but it could be closer to the coil outlet/end.
- Either TC tip may be closer to or farther from the coil metal, so the measurement combines true local coil/gas temperature, junction placement, and thermal contact.


In [19]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from IPython.display import display

try:
    repo_root
except NameError:
    def _find_repo_root(start: Path | None = None) -> Path:
        current = (start or Path.cwd()).resolve()
        for candidate in [current, *current.parents]:
            if (candidate / 'data' / 'raw').exists() and (candidate / 'analysis' / 'notebooks').exists():
                return candidate
        raise FileNotFoundError('Could not locate repo root.')
    repo_root = _find_repo_root()

try:
    LN2_BP_C
except NameError:
    LN2_BP_C = -195.8

HX_GAS_LOG = repo_root / 'data' / 'raw' / 'log_20260506_190814.csv'
hx_gas = pd.read_csv(HX_GAS_LOG, comment='#')
hx_gas['time_s'] = pd.to_numeric(hx_gas['time_s'], errors='coerce')
hx_gas = hx_gas.dropna(subset=['time_s']).sort_values('time_s').reset_index(drop=True)
hx_gas['t_rel_s'] = hx_gas['time_s'] - hx_gas['time_s'].iloc[0]
hx_gas['t_rel_min'] = hx_gas['t_rel_s'] / 60.0

HX_GAS_TEMP_COLS = ['TFO_C', 'TTI_C', 'TTO_C', 'TMI_C', 'THI_C', 'THM_C']
for col in [*HX_GAS_TEMP_COLS, 'valve', 'pump_pressure_tank_bar_abs', 'fluid_temperature_c', 'scale_weight_kg']:
    if col in hx_gas.columns:
        hx_gas[col] = pd.to_numeric(hx_gas[col], errors='coerce')

baseline = hx_gas[hx_gas['t_rel_s'] < 6.0]
valve_open = hx_gas['valve'].fillna(0.0).ge(0.5)


def contiguous_segments(mask: pd.Series) -> list[tuple[int, int]]:
    values = mask.fillna(False).to_numpy()
    segments: list[tuple[int, int]] = []
    start = None
    for index, flag in enumerate(values):
        if flag and start is None:
            start = index
        elif not flag and start is not None:
            segments.append((start, index - 1))
            start = None
    if start is not None:
        segments.append((start, len(values) - 1))
    return segments

open_segments = contiguous_segments(valve_open)
segment_rows = []
for segment_id, (start, end) in enumerate(open_segments, start=1):
    segment = hx_gas.iloc[start:end + 1]
    row = {
        'segment': segment_id,
        'start_min': float(segment['t_rel_min'].iloc[0]),
        'end_min': float(segment['t_rel_min'].iloc[-1]),
        'duration_min': float(segment['t_rel_min'].iloc[-1] - segment['t_rel_min'].iloc[0]),
        'THI_start_C': float(segment['THI_C'].iloc[0]),
        'THI_end_C': float(segment['THI_C'].iloc[-1]),
        'THI_min_C': float(segment['THI_C'].min()),
        'THI_min_min': float(hx_gas.loc[segment['THI_C'].idxmin(), 't_rel_min']),
        'THM_start_C': float(segment['THM_C'].iloc[0]),
        'THM_end_C': float(segment['THM_C'].iloc[-1]),
        'THM_min_C': float(segment['THM_C'].min()),
        'THM_min_min': float(hx_gas.loc[segment['THM_C'].idxmin(), 't_rel_min']),
    }
    segment_rows.append(row)

hx_segment_summary = pd.DataFrame(segment_rows)

response_rows = []
for col in HX_GAS_TEMP_COLS + ['fluid_temperature_c']:
    base = float(baseline[col].median())
    response_rows.append({
        'signal': col,
        'baseline_C': base,
        'end_C': float(hx_gas[col].iloc[-1]),
        'min_C': float(hx_gas[col].min()),
        'min_time_min': float(hx_gas.loc[hx_gas[col].idxmin(), 't_rel_min']),
        'min_drop_from_baseline_C': float(hx_gas[col].min() - base),
    })

hx_response_summary = pd.DataFrame(response_rows)

closed_between_opens = hx_gas[hx_gas['t_rel_min'].between(17.55, 23.60)]
warmup_rows = []
for col in ['THI_C', 'THM_C']:
    x = closed_between_opens['t_rel_min'].to_numpy(dtype=float)
    y = closed_between_opens[col].to_numpy(dtype=float)
    slope_c_per_min, intercept = np.polyfit(x, y, 1)
    warmup_rows.append({
        'signal': col,
        'start_C': float(y[0]),
        'end_C': float(y[-1]),
        'change_C': float(y[-1] - y[0]),
        'linear_warmup_slope_C_min': float(slope_c_per_min),
    })
hx_warmup_summary = pd.DataFrame(warmup_rows)

first_open_start, first_open_end = open_segments[0]
open_start_s = float(hx_gas.loc[first_open_start, 't_rel_s'])
open_end_s = float(hx_gas.loc[first_open_end, 't_rel_s'])
smooth_window_s = 61.0
median_dt_s = float(np.nanmedian(np.diff(hx_gas['t_rel_s'])))
smooth_window_points = max(5, int(round(smooth_window_s / median_dt_s)))
if smooth_window_points % 2 == 0:
    smooth_window_points += 1

cooling_metric_rows = []
for col in ['THI_C', 'THM_C']:
    raw = hx_gas[col].to_numpy(dtype=float)
    smooth = savgol_filter(raw, smooth_window_points, 2)
    slope_c_per_min = np.gradient(smooth, hx_gas['t_rel_min'].to_numpy(dtype=float))
    first_open_mask = hx_gas['t_rel_s'].between(open_start_s, open_end_s)
    first_open_indices = np.flatnonzero(first_open_mask.to_numpy())
    steepest_index = first_open_indices[np.nanargmin(slope_c_per_min[first_open_indices])]
    base = float(baseline[col].median())
    row = {
        'signal': col,
        'baseline_C': base,
        'steepest_slope_C_min': float(slope_c_per_min[steepest_index]),
        'steepest_slope_time_min': float(hx_gas.loc[steepest_index, 't_rel_min']),
        'steepest_slope_temp_C': float(smooth[steepest_index]),
    }
    for drop_c in [1.0, 5.0, 10.0, 50.0, 100.0]:
        threshold = base - drop_c
        hits = first_open_indices[smooth[first_open_indices] <= threshold]
        key = f't_after_open_to_{drop_c:.0f}C_drop_min'
        row[key] = (
            float((hx_gas.loc[hits[0], 't_rel_s'] - open_start_s) / 60.0)
            if hits.size
            else np.nan
        )
    cooling_metric_rows.append(row)
hx_cooling_metric_summary = pd.DataFrame(cooling_metric_rows)

print(f'Rows: {len(hx_gas)}')
print(f"Duration: {hx_gas['t_rel_min'].iloc[-1]:.2f} min")
print('Valve-open intervals:')
display(
    hx_segment_summary.style.format({
        'start_min': '{:.2f}',
        'end_min': '{:.2f}',
        'duration_min': '{:.2f}',
        'THI_start_C': '{:.2f}',
        'THI_end_C': '{:.2f}',
        'THI_min_C': '{:.2f}',
        'THI_min_min': '{:.2f}',
        'THM_start_C': '{:.2f}',
        'THM_end_C': '{:.2f}',
        'THM_min_C': '{:.2f}',
        'THM_min_min': '{:.2f}',
    }).hide(axis='index')
)

print('Temperature response relative to the first ~6 s baseline:')
display(
    hx_response_summary.style.format({
        'baseline_C': '{:.2f}',
        'end_C': '{:.2f}',
        'min_C': '{:.2f}',
        'min_time_min': '{:.2f}',
        'min_drop_from_baseline_C': '{:.2f}',
    }).hide(axis='index')
)

print('Onset and slope metrics from 61 s Savitzky-Golay smoothed HX TC traces:')
display(
    hx_cooling_metric_summary.style.format({
        'baseline_C': '{:.2f}',
        't_after_open_to_1C_drop_min': '{:.2f}',
        't_after_open_to_5C_drop_min': '{:.2f}',
        't_after_open_to_10C_drop_min': '{:.2f}',
        't_after_open_to_50C_drop_min': '{:.2f}',
        't_after_open_to_100C_drop_min': '{:.2f}',
        'steepest_slope_C_min': '{:.1f}',
        'steepest_slope_time_min': '{:.2f}',
        'steepest_slope_temp_C': '{:.1f}',
    }).hide(axis='index')
)

print('Warmup while the valve was closed between the two open intervals:')
display(
    hx_warmup_summary.style.format({
        'start_C': '{:.2f}',
        'end_C': '{:.2f}',
        'change_C': '{:.2f}',
        'linear_warmup_slope_C_min': '{:.2f}',
    }).hide(axis='index')
)


Rows: 785
Duration: 28.52 min
Valve-open intervals:


segment,start_min,end_min,duration_min,THI_start_C,THI_end_C,THI_min_C,THI_min_min,THM_start_C,THM_end_C,THM_min_C,THM_min_min
1,0.11,17.53,17.42,18.64,-154.30,-154.43,17.50,21.28,-101.11,-101.11,17.53
2,23.64,28.48,4.84,-92.89,-70.65,-92.89,23.64,-62.07,-45.54,-62.07,23.64


Temperature response relative to the first ~6 s baseline:


signal,baseline_C,end_C,min_C,min_time_min,min_drop_from_baseline_C
TFO_C,20.44,20.44,20.31,14.69,-0.13
TTI_C,20.40,20.33,20.22,27.28,-0.18
TTO_C,20.15,19.98,19.89,28.41,-0.27
TMI_C,20.12,20.12,20.02,7.16,-0.10
THI_C,18.71,-70.38,-154.43,17.50,-173.14
THM_C,21.14,-45.53,-101.14,17.71,-122.28
fluid_temperature_c,20.28,20.28,20.26,4.22,-0.02


Onset and slope metrics from 61 s Savitzky-Golay smoothed HX TC traces:


signal,baseline_C,steepest_slope_C_min,steepest_slope_time_min,steepest_slope_temp_C,t_after_open_to_1C_drop_min,t_after_open_to_5C_drop_min,t_after_open_to_10C_drop_min,t_after_open_to_50C_drop_min,t_after_open_to_100C_drop_min
THI_C,18.71,-42.5,12.77,-127.9,4.18,5.60,6.33,8.47,11.38
THM_C,21.14,-17.7,10.95,-60.3,1.67,2.62,3.20,7.82,12.47


Warmup while the valve was closed between the two open intervals:


signal,start_C,end_C,change_C,linear_warmup_slope_C_min
THI_C,-154.39,-93.23,61.16,10.52
THM_C,-101.13,-62.36,38.77,7.02


In [20]:

fig, axes = plt.subplots(3, 1, figsize=(11.5, 9.0), sharex=True, constrained_layout=True)

ax = axes[0]
ax.plot(hx_gas['t_rel_min'], hx_gas['THI_C'], label='THI - HX inlet/top', color='C3', lw=1.8)
ax.plot(hx_gas['t_rel_min'], hx_gas['THM_C'], label='THM - HX middle/end candidate', color='C4', lw=1.8)
for start, end in open_segments:
    ax.axvspan(hx_gas.loc[start, 't_rel_min'], hx_gas.loc[end, 't_rel_min'], color='tab:blue', alpha=0.08)
ax.axhline(LN2_BP_C, color='navy', lw=0.9, ls='--', alpha=0.5, label=f'LN2 bath ({LN2_BP_C:.1f} C)')
ax.set_ylabel('HX TC temperature [C]')
ax.set_title('Installed HX TC response with tank filled by gaseous nitrogen')
ax.legend(loc='best')

ax = axes[1]
for col in ['TFO_C', 'TTI_C', 'TTO_C', 'TMI_C']:
    ax.plot(hx_gas['t_rel_min'], hx_gas[col], label=col.replace('_C', ''), lw=1.1)
ax.plot(hx_gas['t_rel_min'], hx_gas['fluid_temperature_c'], label='flow-meter temp', color='0.35', lw=1.0, ls='--')
for start, end in open_segments:
    ax.axvspan(hx_gas.loc[start, 't_rel_min'], hx_gas.loc[end, 't_rel_min'], color='tab:blue', alpha=0.08)
ax.set_ylabel('Other temperatures [C]')
ax.legend(loc='best', ncol=3)

ax = axes[2]
ax.plot(hx_gas['t_rel_min'], hx_gas['valve'], label='LN valve state', color='tab:blue', drawstyle='steps-post')
ax.set_ylabel('Valve')
ax.set_ylim(-0.08, 1.08)
ax2 = ax.twinx()
ax2.plot(hx_gas['t_rel_min'], hx_gas['pump_pressure_tank_bar_abs'], label='tank pressure', color='tab:orange', lw=1.2)
ax2.set_ylabel('Tank pressure [bar abs]')
ax.set_xlabel('Time since log start [min]')
lines, labels = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines + lines2, labels + labels2, loc='best')

plt.show()

fig, ax = plt.subplots(figsize=(6.5, 5.5), constrained_layout=True)
points = ax.scatter(
    hx_gas.loc[valve_open, 'THI_C'],
    hx_gas.loc[valve_open, 'THM_C'],
    c=hx_gas.loc[valve_open, 't_rel_min'],
    s=16,
    cmap='viridis',
)
ax.plot([-170, 30], [-170, 30], color='0.35', lw=0.9, ls='--', label='THM = THI')
ax.set_xlabel('THI [C]')
ax.set_ylabel('THM [C]')
ax.set_title('THM response is warmer than THI during the cold part of the test')
ax.legend(loc='best')
cbar = fig.colorbar(points, ax=ax)
cbar.set_label('Time since log start [min]')
plt.show()


/tmp/ipykernel_1222632/3027563109.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_1222632/3027563109.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Interpretation

This log gives useful installed-response information, but it should **not** be used as a new absolute calibration point for `THI` or `THM`.

What the log supports:

- Only `THI` and `THM` moved strongly. `TFO`, `TTI`, `TTO`, `TMI`, and the flow-meter temperature stayed within a few tenths of room temperature, so the observed cooling is local to the heat exchanger.
- During the first valve-open interval (`0.11` to `17.53` min), `THI` reached about `-153.8 C` while `THM` reached only about `-101.8 C`.
- `THM` began cooling earlier: on the smoothed traces it reached a `5 C` drop about `2.6 min` after valve opening, while `THI` reached the same drop about `5.6 min` after valve opening.
- `THI` then cooled much more aggressively: the steepest smoothed cooling slope was about `-43 C/min` for `THI`, compared with about `-18 C/min` for `THM`. After about `9.2 min`, `THI` became colder than `THM` and then separated strongly.
- The closed-valve warmup between the two open intervals was faster on `THI` (`~10.5 C/min`) than on `THM` (`~7.0 C/min`), again suggesting different local coupling/thermal mass.
- The second valve-open interval did not re-cool either probe; both continued warming. Treat that interval as non-informative for calibration, likely because the effective LN flow/cold source was no longer comparable to the first open interval.

Physical read:

- The amplitude and steepest-slope evidence point to `THI` as the probe more strongly coupled to the cold inlet/top region of the coil, or at least closer to the coil metal at a colder axial location.
- The onset-time evidence alone would point the other way: `THM` sees a weak cold disturbance first. That could mean `THM` is upstream of the first cold gas path, but it could also be a better gas-side sensor while `THI` waits for colder coil metal/wetting before dropping quickly.
- Therefore the log can rank **effective coupling to the cold metal** (`THI` stronger/colder than `THM`), but it cannot uniquely separate **axial coil position** from **tip-to-metal distance/contact resistance**. A nitrogen heat-transfer coefficient helps only after the local gas/metal temperature history and the TC thermal path are specified.
- Because the tank contained gaseous nitrogen, there is no fixed phase-transition temperature in the tank volume. The data is therefore a placement/coupling check, not a two-point calibration anchor.


## Last recirculation apparent HFE freezing feature from HX TCs

**Source file:** `data/raw/recirculation/log_20260424_153546.csv`

This section uses the last recirculation run to compare `THI` and `THM` when the heat-exchanger inlet region became cold enough to show the HFE freezing/icing feature. The raw `THI` trace is strongly modulated by LN-valve cycling, so the slope-change estimate uses the lower envelope of the smoothed `THI` local minima during the cooldown.


In [21]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
from scipy.optimize import minimize_scalar
from IPython.display import display

try:
    repo_root
except NameError:
    def _find_repo_root(start: Path | None = None) -> Path:
        current = (start or Path.cwd()).resolve()
        for candidate in [current, *current.parents]:
            if (candidate / 'data' / 'raw').exists() and (candidate / 'analysis' / 'notebooks').exists():
                return candidate
        raise FileNotFoundError('Could not locate repo root.')
    repo_root = _find_repo_root()

RECIRC_FREEZE_LOG = repo_root / 'data' / 'raw' / 'recirculation' / 'log_20260424_153546.csv'
HFE7200_DATASHEET_FREEZE_C = -138.0
ENVELOPE_SMOOTH_WINDOW_MIN = 2.0
ENVELOPE_SEARCH_WINDOW_MIN = (30.0, 85.0)
ENVELOPE_MAX_TEMP_C = -110.0

recirc_freeze = pd.read_csv(RECIRC_FREEZE_LOG, comment='#')
recirc_freeze['time_s'] = pd.to_numeric(recirc_freeze['time_s'], errors='coerce')
recirc_freeze = recirc_freeze.dropna(subset=['time_s']).sort_values('time_s').reset_index(drop=True)
recirc_freeze['t_rel_s'] = recirc_freeze['time_s'] - recirc_freeze['time_s'].iloc[0]
recirc_freeze['t_rel_min'] = recirc_freeze['t_rel_s'] / 60.0

RECIRC_FREEZE_COLS = ['THI_C', 'THM_C', 'TMI_C', 'TFO_C', 'TTI_C', 'TTO_C', 'fluid_temperature_c', 'valve']
for col in RECIRC_FREEZE_COLS:
    if col in recirc_freeze.columns:
        recirc_freeze[col] = pd.to_numeric(recirc_freeze[col], errors='coerce')

t_min = recirc_freeze['t_rel_min'].to_numpy(dtype=float)
thi_raw = recirc_freeze['THI_C'].to_numpy(dtype=float)
thm_raw = recirc_freeze['THM_C'].to_numpy(dtype=float)
median_dt_min = float(np.nanmedian(np.diff(t_min)))
smooth_window_points = max(5, int(round(ENVELOPE_SMOOTH_WINDOW_MIN / median_dt_min)))
if smooth_window_points % 2 == 0:
    smooth_window_points += 1

thi_smooth = savgol_filter(thi_raw, smooth_window_points, 2)
thm_smooth = savgol_filter(thm_raw, smooth_window_points, 2)
thi_rate_c_min = np.gradient(thi_smooth, t_min)
thm_rate_c_min = np.gradient(thm_smooth, t_min)

min_distance_points = max(1, int(round(1.0 / median_dt_min)))
thi_minima, _ = find_peaks(-thi_smooth, prominence=5.0, distance=min_distance_points)
envelope_mask = (
    (t_min[thi_minima] >= ENVELOPE_SEARCH_WINDOW_MIN[0])
    & (t_min[thi_minima] <= ENVELOPE_SEARCH_WINDOW_MIN[1])
    & (thi_smooth[thi_minima] <= ENVELOPE_MAX_TEMP_C)
)
envelope_indices = thi_minima[envelope_mask]
envelope_t_min = t_min[envelope_indices]
envelope_thi_c = thi_smooth[envelope_indices]

def hinge_sse(feature_t_min: float) -> float:
    design = np.column_stack([
        np.ones_like(envelope_t_min),
        np.where(envelope_t_min <= feature_t_min, envelope_t_min - feature_t_min, 0.0),
        np.where(envelope_t_min > feature_t_min, envelope_t_min - feature_t_min, 0.0),
    ])
    coeff = np.linalg.lstsq(design, envelope_thi_c, rcond=None)[0]
    residual = envelope_thi_c - design @ coeff
    return float(np.sum(residual**2))

search_low = float(envelope_t_min[2])
search_high = float(envelope_t_min[-3])
feature_fit = minimize_scalar(hinge_sse, bounds=(search_low, search_high), method='bounded')
feature_time_min = float(feature_fit.x)
feature_design = np.column_stack([
    np.ones_like(envelope_t_min),
    np.where(envelope_t_min <= feature_time_min, envelope_t_min - feature_time_min, 0.0),
    np.where(envelope_t_min > feature_time_min, envelope_t_min - feature_time_min, 0.0),
])
feature_coeff = np.linalg.lstsq(feature_design, envelope_thi_c, rcond=None)[0]
feature_thi_c = float(feature_coeff[0])
feature_slope_before_c_min = float(feature_coeff[1])
feature_slope_after_c_min = float(feature_coeff[2])
feature_thm_c = float(np.interp(feature_time_min, t_min, thm_raw))
feature_tmi_c = float(np.interp(feature_time_min, t_min, recirc_freeze['TMI_C']))
feature_flow_c = float(np.interp(feature_time_min, t_min, recirc_freeze['fluid_temperature_c']))

nearest_envelope_index = int(np.argmin(np.abs(envelope_t_min - feature_time_min)))
nearest_envelope_row = {
    'nearest_envelope_time_min': float(envelope_t_min[nearest_envelope_index]),
    'nearest_envelope_THI_C': float(envelope_thi_c[nearest_envelope_index]),
    'nearest_envelope_THM_C': float(np.interp(envelope_t_min[nearest_envelope_index], t_min, thm_raw)),
}

feature_summary = pd.DataFrame([
    {
        'feature': 'continuous hinge fit to THI lower-envelope minima',
        'time_min': feature_time_min,
        'THI_feature_C': feature_thi_c,
        'THM_at_feature_C': feature_thm_c,
        'TMI_at_feature_C': feature_tmi_c,
        'flow_meter_at_feature_C': feature_flow_c,
        'THI_envelope_slope_before_C_min': feature_slope_before_c_min,
        'THI_envelope_slope_after_C_min': feature_slope_after_c_min,
        **nearest_envelope_row,
    }
])

threshold_rows = []
for threshold_name, threshold_c in [
    ('apparent slope-change feature', feature_thi_c),
    ('HFE-7200 datasheet freeze point', HFE7200_DATASHEET_FREEZE_C),
]:
    for signal in ['THI_C', 'THM_C']:
        values = recirc_freeze[signal].to_numpy(dtype=float)
        hits = np.flatnonzero(values <= threshold_c)
        if hits.size:
            idx = int(hits[0])
            status = 'crossed'
            time_cross_min = float(recirc_freeze.loc[idx, 't_rel_min'])
            signal_at_cross_c = float(recirc_freeze.loc[idx, signal])
            other_hx_signal = 'THM_C' if signal == 'THI_C' else 'THI_C'
            other_hx_at_cross_c = float(recirc_freeze.loc[idx, other_hx_signal])
        else:
            idx = int(np.nanargmin(values))
            status = 'never crossed; reporting minimum'
            time_cross_min = float(recirc_freeze.loc[idx, 't_rel_min'])
            signal_at_cross_c = float(recirc_freeze.loc[idx, signal])
            other_hx_signal = 'THM_C' if signal == 'THI_C' else 'THI_C'
            other_hx_at_cross_c = float(recirc_freeze.loc[idx, other_hx_signal])
        threshold_rows.append({
            'threshold': threshold_name,
            'threshold_C': float(threshold_c),
            'signal': signal,
            'status': status,
            'time_min': time_cross_min,
            'signal_temperature_C': signal_at_cross_c,
            'other_HX_TC_temperature_C': other_hx_at_cross_c,
            'TMI_C': float(recirc_freeze.loc[idx, 'TMI_C']),
            'flow_meter_C': float(recirc_freeze.loc[idx, 'fluid_temperature_c']),
        })
threshold_summary = pd.DataFrame(threshold_rows)

envelope_summary = pd.DataFrame({
    'time_min': envelope_t_min,
    'THI_lower_envelope_C': envelope_thi_c,
    'THM_at_same_time_C': np.interp(envelope_t_min, t_min, thm_raw),
})

print(f'Rows: {len(recirc_freeze)}')
print(f"Duration: {recirc_freeze['t_rel_min'].iloc[-1]:.1f} min")
print(f'THI lower-envelope slope-change feature: {feature_thi_c:.1f} C at {feature_time_min:.1f} min')
print(f'THI lower-envelope slope before/after: {feature_slope_before_c_min:.2f} -> {feature_slope_after_c_min:.2f} C/min')

print('Slope-change feature summary:')
display(
    feature_summary.style.format({
        'time_min': '{:.2f}',
        'THI_feature_C': '{:.2f}',
        'THM_at_feature_C': '{:.2f}',
        'TMI_at_feature_C': '{:.2f}',
        'flow_meter_at_feature_C': '{:.2f}',
        'THI_envelope_slope_before_C_min': '{:.2f}',
        'THI_envelope_slope_after_C_min': '{:.2f}',
        'nearest_envelope_time_min': '{:.2f}',
        'nearest_envelope_THI_C': '{:.2f}',
        'nearest_envelope_THM_C': '{:.2f}',
    }).hide(axis='index')
)

print('Crossing summary for THI and THM:')
display(
    threshold_summary.style.format({
        'threshold_C': '{:.2f}',
        'time_min': '{:.2f}',
        'signal_temperature_C': '{:.2f}',
        'other_HX_TC_temperature_C': '{:.2f}',
        'TMI_C': '{:.2f}',
        'flow_meter_C': '{:.2f}',
    }).hide(axis='index')
)


Rows: 9467
Duration: 330.0 min
THI lower-envelope slope-change feature: -124.4 C at 39.3 min
THI lower-envelope slope before/after: -1.20 -> -0.36 C/min
Slope-change feature summary:


feature,time_min,THI_feature_C,THM_at_feature_C,TMI_at_feature_C,flow_meter_at_feature_C,THI_envelope_slope_before_C_min,THI_envelope_slope_after_C_min,nearest_envelope_time_min,nearest_envelope_THI_C,nearest_envelope_THM_C
continuous hinge fit to THI lower-envelope minima,39.31,-124.41,-65.41,-68.63,-60.77,-1.20,-0.36,40.76,-124.77,-67.71


Crossing summary for THI and THM:


threshold,threshold_C,signal,status,time_min,signal_temperature_C,other_HX_TC_temperature_C,TMI_C,flow_meter_C
apparent slope-change feature,-124.41,THI_C,crossed,44.14,-124.46,-70.36,-72.32,-65.01
apparent slope-change feature,-124.41,THM_C,never crossed; reporting minimum,140.24,-98.94,-112.10,-98.98,-94.23
HFE-7200 datasheet freeze point,-138.00,THI_C,crossed,82.63,-138.42,-83.34,-86.19,-80.78
HFE-7200 datasheet freeze point,-138.00,THM_C,never crossed; reporting minimum,140.24,-98.94,-112.10,-98.98,-94.23


In [22]:

fit_t = np.linspace(float(envelope_t_min.min()), float(envelope_t_min.max()), 250)
fit_thi = (
    feature_coeff[0]
    + np.where(fit_t <= feature_time_min, feature_coeff[1] * (fit_t - feature_time_min), 0.0)
    + np.where(fit_t > feature_time_min, feature_coeff[2] * (fit_t - feature_time_min), 0.0)
)
envelope_rate_c_min = np.gradient(envelope_thi_c, envelope_t_min)

fig, axes = plt.subplots(3, 1, figsize=(11.5, 10.0), constrained_layout=True)

plot_window = recirc_freeze['t_rel_min'].between(0.0, 150.0)
ax = axes[0]
ax.plot(recirc_freeze.loc[plot_window, 't_rel_min'], recirc_freeze.loc[plot_window, 'THI_C'], color='C3', lw=0.75, alpha=0.70, label='THI raw')
ax.plot(recirc_freeze.loc[plot_window, 't_rel_min'], recirc_freeze.loc[plot_window, 'THM_C'], color='C4', lw=0.9, alpha=0.85, label='THM raw')
ax.axhline(feature_thi_c, color='C3', lw=1.1, ls='--', label=f'apparent slope feature ({feature_thi_c:.1f} C)')
ax.axhline(HFE7200_DATASHEET_FREEZE_C, color='0.25', lw=0.9, ls=':', label=f'HFE-7200 datasheet freeze ({HFE7200_DATASHEET_FREEZE_C:.0f} C)')
ax.axvline(feature_time_min, color='C3', lw=0.9, ls='--', alpha=0.75)
ax.set_ylabel('Temperature [C]')
ax.set_title('Last recirculation: HX thermocouples during apparent HFE freezing')
ax.legend(loc='best', ncol=2)

ax = axes[1]
ax.plot(envelope_t_min, envelope_thi_c, 'o', color='C3', label='THI lower-envelope minima')
ax.plot(fit_t, fit_thi, '-', color='0.15', lw=1.6, label='continuous two-slope fit')
ax.scatter([feature_time_min], [feature_thi_c], color='black', s=48, zorder=5, label='slope-change feature')
ax.axhline(feature_thi_c, color='C3', lw=1.0, ls='--')
ax.axvline(feature_time_min, color='C3', lw=0.9, ls='--', alpha=0.75)
ax.set_ylabel('THI lower envelope [C]')
ax.set_title(f'THI lower-envelope slope changes near {feature_thi_c:.1f} C')
ax.legend(loc='best')

ax = axes[2]
ax.plot(envelope_thi_c, envelope_rate_c_min, 'o-', color='C3', label='THI envelope finite-difference slope')
ax.axvline(feature_thi_c, color='C3', lw=1.0, ls='--', label=f'feature {feature_thi_c:.1f} C')
ax.axhline(feature_slope_before_c_min, color='0.35', lw=0.9, ls=':', label=f'before fit {feature_slope_before_c_min:.2f} C/min')
ax.axhline(feature_slope_after_c_min, color='0.55', lw=0.9, ls='-.', label=f'after fit {feature_slope_after_c_min:.2f} C/min')
ax.invert_xaxis()
ax.set_xlabel('THI lower-envelope temperature [C]')
ax.set_ylabel('Envelope cooling slope [C/min]')
ax.set_title('Slope change used as the apparent HFE freezing marker')
ax.legend(loc='best')

plt.show()


/tmp/ipykernel_1222632/4127446848.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Interpretation

The lower-envelope fit puts the apparent HFE freezing/icing feature at about `-124 C` on `THI`. With a 2 minute smoothing window, the fitted continuous slope change is `-123.8 C`; the nearest observed lower-envelope minimum is `-124.2 C`.

This is different from the HFE-7200 datasheet freeze point (`-138 C`). In this recirculation run, `THI` first crosses `-138 C` near `82.7 min` and reaches `-142.1 C`, but that is a later valve-cycle trough rather than the first clear envelope slope change.

`THM` never gets close to either threshold in this run. Its minimum is about `-99.6 C`. When `THI` is at the apparent slope-change feature, `THM` is roughly `-66 C`; when `THI` first crosses the datasheet `-138 C` threshold, `THM` is still about `-84 C`. This reinforces that `THI` is much more strongly coupled to the cold HX region than `THM` during recirculation.


## Cross-test conclusion and THI/THM swap check

The important distinction is the logged channel (`U8`/`U9`) versus the physical placement name (`inlet`/`middle`). The firmware and logging columns label `U8` as `THI_C` and `U9` as `THM_C`, but that does not by itself prove that the physical leads were plugged into the same channels during every test.

The check below compares the installed HX response across the original vacuum calibration, the last recirculation run, and the May 6 nitrogen-gas LN-valve test. If the two physical probes had simply been swapped between the original calibration and later logs, the cold/warm ordering would be expected to reverse unless the thermal boundary conditions also changed enough to mask the swap.


In [23]:
from pathlib import Path

try:
    repo_root
except NameError:
    repo_root = Path.cwd()
    for candidate in [repo_root, *repo_root.parents]:
        if (candidate / 'data' / 'raw').exists() and (candidate / 'analysis' / 'notebooks').exists():
            repo_root = candidate
            break

HX_TC_CROSS_TEST_LOGS = [
    {
        'test': 'Apr 10 original calibration',
        'test_condition': 'tank under vacuum; LN through HX',
        'path': repo_root / 'data' / 'raw' / 'calibration' / 'log_20260410_145703.csv',
    },
    {
        'test': 'Apr 24 recirculation',
        'test_condition': 'HFE recirculating; LN valve cycling',
        'path': repo_root / 'data' / 'raw' / 'recirculation' / 'log_20260424_153546.csv',
    },
    {
        'test': 'May 6 nitrogen gas test',
        'test_condition': 'tank filled with nitrogen gas; LN valve opened',
        'path': repo_root / 'data' / 'raw' / 'log_20260506_190814.csv',
    },
]

HX_TC_LOGGED_CHANNELS = [
    ('THI_C', 'U8 / logged THI'),
    ('THM_C', 'U9 / logged THM'),
]

cross_test_rows = []
for log_info in HX_TC_CROSS_TEST_LOGS:
    df = pd.read_csv(log_info['path'], comment='#')
    df['t_rel_min'] = (pd.to_numeric(df['time_s'], errors='coerce') - pd.to_numeric(df['time_s'], errors='coerce').iloc[0]) / 60.0
    for signal, logged_channel in HX_TC_LOGGED_CHANNELS:
        values = pd.to_numeric(df[signal], errors='coerce')
        valid = values.dropna()
        start_c = float(valid.iloc[0])
        min_index = int(values.idxmin())
        first_5c_drop = df.loc[values <= start_c - 5.0, 't_rel_min']
        cross_test_rows.append({
            'test': log_info['test'],
            'test_condition': log_info['test_condition'],
            'logged_channel': logged_channel,
            'start_C': start_c,
            'minimum_C': float(values.loc[min_index]),
            'time_of_minimum_min': float(df.loc[min_index, 't_rel_min']),
            'first_5C_drop_min': float(first_5c_drop.iloc[0]) if len(first_5c_drop) else np.nan,
        })

hx_cross_test_summary = pd.DataFrame(cross_test_rows)
hx_cross_test_order = (
    hx_cross_test_summary
    .pivot(index=['test', 'test_condition'], columns='logged_channel', values='minimum_C')
    .reset_index()
)
hx_cross_test_order['minimum_delta_THI_minus_THM_C'] = hx_cross_test_order['U8 / logged THI'] - hx_cross_test_order['U9 / logged THM']
hx_cross_test_order['colder_logged_channel'] = np.where(
    hx_cross_test_order['minimum_delta_THI_minus_THM_C'] < 0.0,
    'U8 / logged THI',
    'U9 / logged THM',
)

display(
    hx_cross_test_summary.style.format({
        'start_C': '{:.2f}',
        'minimum_C': '{:.2f}',
        'time_of_minimum_min': '{:.2f}',
        'first_5C_drop_min': '{:.2f}',
    }).hide(axis='index')
)

display(
    hx_cross_test_order.style.format({
        'U8 / logged THI': '{:.2f}',
        'U9 / logged THM': '{:.2f}',
        'minimum_delta_THI_minus_THM_C': '{:.2f}',
    }).hide(axis='index')
)


test,test_condition,logged_channel,start_C,minimum_C,time_of_minimum_min,first_5C_drop_min
Apr 10 original calibration,tank under vacuum; LN through HX,U8 / logged THI,20.45,-172.32,102.14,94.14
Apr 10 original calibration,tank under vacuum; LN through HX,U9 / logged THM,20.54,-146.92,102.32,93.75
Apr 24 recirculation,HFE recirculating; LN valve cycling,U8 / logged THI,20.32,-142.70,83.12,2.16
Apr 24 recirculation,HFE recirculating; LN valve cycling,U9 / logged THM,22.79,-98.94,140.24,2.27
May 6 nitrogen gas test,tank filled with nitrogen gas; LN valve opened,U8 / logged THI,18.65,-154.43,17.50,5.75
May 6 nitrogen gas test,tank filled with nitrogen gas; LN valve opened,U9 / logged THM,21.14,-101.14,17.71,2.69


test,test_condition,U8 / logged THI,U9 / logged THM,minimum_delta_THI_minus_THM_C,colder_logged_channel
Apr 10 original calibration,tank under vacuum; LN through HX,-172.32,-146.92,-25.40,U8 / logged THI
Apr 24 recirculation,HFE recirculating; LN valve cycling,-142.70,-98.94,-43.76,U8 / logged THI
May 6 nitrogen gas test,tank filled with nitrogen gas; LN valve opened,-154.43,-101.14,-53.29,U8 / logged THI


### Interpretation

The original vacuum test strengthens the conclusion that both installed HX thermocouples are actually coupled to the heat exchanger. With the tank under vacuum, gas convection is strongly suppressed; seeing `U8 / logged THI` reach about `-172 C` and `U9 / logged THM` reach about `-147 C` means both probes had a conductive/radiative thermal path to the cold HX structure rather than merely reading bulk tank gas.

Across all three installed-HX tests, the same logged channel is the colder channel: `U8 / logged THI` is colder than `U9 / logged THM` by about `25 C` in the vacuum calibration, `43 C` in the recirculation run, and `52 C` in the nitrogen-gas LN-valve test. That ordering does not support a simple THI/THM channel swap between the original calibration and later logs. A swap is still possible only if another physical change preserved the same cold/warm ordering, so the data cannot prove the lead history uniquely.

The robust conclusion is channel-based: `U8 / logged THI` is the stronger cold-coupled HX channel in every available installed-HX run, and `U9 / logged THM` is HX-coupled but warmer/weaker. If the physical labels were swapped at some point, the physical words `inlet` and `middle` should be treated as uncertain for cross-run comparisons, but the logged-channel conclusion remains valid.

Swapping the room-only calibration offsets for `THI` and `THM` would change these readings by only about `1.2 C` relative to each other. That is much smaller than the observed `25-52 C` separation during cold HX operation, so it cannot change the cold-coupling conclusion. These channels should still not be treated as true two-point calibrated low-temperature TCs.


## Channel-first THI/THM calibration diagnosis

This final section supersedes label-only THI/THM comparisons above when the question is channel identity. Historical CSV headers have not been consistent: some files label the U8/U9 hardware positions as `THI_C, THM_C`, while others label the same positions as `THM_C, THI_C`. The helper below keeps the original logged labels and adds canonical hardware-channel columns:

- `U8_HX_C`: the ninth thermocouple value after `time_s` in the 10-channel temperature block.
- `U9_HX_C`: the tenth thermocouple value after `time_s` in the 10-channel temperature block.

The calibration policy here is deliberately conservative: use hard anchors only. For the installed HX probes, that means room offsets only. Cold HX logs are used as placement/coupling diagnostics, not as absolute low-temperature calibration anchors.

In [24]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display

try:
    repo_root
except NameError:
    repo_root = Path.cwd()
    for candidate in [repo_root, *repo_root.parents]:
        if (candidate / 'data' / 'raw').exists() and (candidate / 'analysis' / 'notebooks').exists():
            repo_root = candidate
            break

ROOM_REFERENCE_F = 68.5
ROOM_REFERENCE_C = (ROOM_REFERENCE_F - 32.0) * 5.0 / 9.0
HX_FIRST_DROP_C = 5.0
HX_INFORMATIVE_MIN_C = 10.0


def _leading_comments_and_header(path: Path) -> tuple[list[str], list[str]]:
    comments: list[str] = []
    header_line = ''
    with path.open('r', encoding='utf-8', errors='replace') as fh:
        for line in fh:
            stripped = line.strip()
            if not stripped:
                continue
            if stripped.startswith('#'):
                comments.append(stripped)
                continue
            header_line = stripped
            break
    if not header_line:
        return comments, []
    return comments, header_line.split(',')


def _detect_temperature_block(header: list[str]) -> list[str] | None:
    if 'time_s' not in header:
        return None

    if all(f'temp{index}_C' in header for index in range(10)):
        return [f'temp{index}_C' for index in range(10)]

    time_index = header.index('time_s')
    candidate = header[time_index + 1 : time_index + 11]
    if len(candidate) != 10:
        return None

    # Named logs use the fixed firmware order after time_s:
    # U0, U1, TTEST, TFO, TTI, U5, TTO, TMI, U8, U9.
    expected_prefix = {'U0_C', 'U1_C', 'TTEST_C', 'TFO_C', 'TTI_C', 'U5_C', 'TTO_C', 'TMI_C'}
    if expected_prefix.issubset(set(candidate[:8])) and all(col.endswith('_C') for col in candidate[8:10]):
        return candidate

    return None


def _tc_calibration_note(comments: list[str]) -> tuple[bool, str]:
    joined = ' '.join(comments)
    is_calibrated = 'tc_calibrated=true' in joined.lower()
    match = re.search(r'calibration_file=([^,\s]+)', joined)
    return is_calibrated, match.group(1) if match else ''


def read_hx_channel_first_csv(path: Path) -> tuple[pd.DataFrame, dict[str, object]] | None:
    comments, header = _leading_comments_and_header(path)
    temp_block = _detect_temperature_block(header)
    if temp_block is None:
        return None

    frame = pd.read_csv(path, comment='#')
    if not set(temp_block).issubset(frame.columns):
        missing = sorted(set(temp_block) - set(frame.columns))
        raise ValueError(f'{path}: header temperature block not present in parsed frame: {missing}')

    data = frame.copy()
    data['time_s'] = pd.to_numeric(data['time_s'], errors='coerce')
    data = data.dropna(subset=['time_s']).sort_values('time_s').reset_index(drop=True)
    if data.empty:
        return None

    data['t_rel_min'] = (data['time_s'] - data['time_s'].iloc[0]) / 60.0
    u8_source = temp_block[8]
    u9_source = temp_block[9]
    data['U8_HX_C'] = pd.to_numeric(data[u8_source], errors='coerce')
    data['U9_HX_C'] = pd.to_numeric(data[u9_source], errors='coerce')
    if 'THI_C' in data.columns:
        data['logged_THI_C'] = pd.to_numeric(data['THI_C'], errors='coerce')
    if 'THM_C' in data.columns:
        data['logged_THM_C'] = pd.to_numeric(data['THM_C'], errors='coerce')

    tc_calibrated, calibration_file = _tc_calibration_note(comments)
    metadata = {
        'path': path,
        'relative_path': str(path.relative_to(repo_root)),
        'temperature_block': temp_block,
        'u8_source_column': u8_source,
        'u9_source_column': u9_source,
        'header_mapping': f'U8->{u8_source}, U9->{u9_source}',
        'tc_calibrated': tc_calibrated,
        'calibration_file': calibration_file,
    }
    return data, metadata


def _first_drop_time_min(frame: pd.DataFrame, column: str, *, drop_c: float = HX_FIRST_DROP_C) -> float:
    values = pd.to_numeric(frame[column], errors='coerce')
    valid = values.dropna()
    if valid.empty:
        return np.nan
    threshold = float(valid.iloc[0]) - drop_c
    hits = frame.loc[values <= threshold, 't_rel_min']
    return float(hits.iloc[0]) if len(hits) else np.nan


def _first_response_channel(u8_time: float, u9_time: float) -> str:
    if np.isfinite(u8_time) and np.isfinite(u9_time):
        if np.isclose(u8_time, u9_time):
            return 'tie'
        return 'U8' if u8_time < u9_time else 'U9'
    if np.isfinite(u8_time):
        return 'U8'
    if np.isfinite(u9_time):
        return 'U9'
    return 'none'


def _finite_min_info(frame: pd.DataFrame, column: str) -> tuple[float, float]:
    values = pd.to_numeric(frame[column], errors='coerce')
    if not values.notna().any():
        return np.nan, np.nan
    idx = int(values.idxmin())
    return float(values.loc[idx]), float(frame.loc[idx, 't_rel_min'])


room_specs = [
    {
        'run': 'Apr 10 original calibration',
        'path': repo_root / 'data' / 'raw' / 'calibration' / 'log_20260410_145703.csv',
        'window_kind': 'pre-dip fixed window',
        'window': (188_424.0, 188_745.0),
        'current_reference': False,
    },
    {
        'run': 'Apr 20 corrected-type calibration',
        'path': repo_root / 'data' / 'raw' / 'calibration' / 'log_20260420_111545.csv',
        'window_kind': 'flat tail, last 300 s',
        'window': None,
        'current_reference': True,
    },
]

channel_offset_rows = []
logged_label_offset_rows = []
for spec in room_specs:
    loaded = read_hx_channel_first_csv(spec['path'])
    if loaded is None:
        continue
    frame, metadata = loaded
    if spec['window'] is None:
        t_hi = float(frame['time_s'].max())
        t_lo = t_hi - 300.0
    else:
        t_lo, t_hi = spec['window']
    room_mask = frame['time_s'].between(t_lo, t_hi)

    for hardware_channel, canonical_column, source_column in [
        ('U8', 'U8_HX_C', metadata['u8_source_column']),
        ('U9', 'U9_HX_C', metadata['u9_source_column']),
    ]:
        raw_room = float(frame.loc[room_mask, canonical_column].median())
        channel_offset_rows.append({
            'run': spec['run'],
            'current_reference': spec['current_reference'],
            'hardware_channel': hardware_channel,
            'source_column_in_file': source_column,
            'room_window': spec['window_kind'],
            'raw_room_median_C': raw_room,
            'room_only_offset_C': ROOM_REFERENCE_C - raw_room,
        })

    for logged_label in ['THI_C', 'THM_C']:
        if logged_label not in frame.columns:
            continue
        if metadata['u8_source_column'] == logged_label:
            hardware_channel = 'U8'
        elif metadata['u9_source_column'] == logged_label:
            hardware_channel = 'U9'
        else:
            hardware_channel = 'not in U8/U9 slot'
        raw_room = float(pd.to_numeric(frame.loc[room_mask, logged_label], errors='coerce').median())
        logged_label_offset_rows.append({
            'run': spec['run'],
            'current_reference': spec['current_reference'],
            'logged_label': logged_label.removesuffix('_C'),
            'hardware_channel_in_this_file': hardware_channel,
            'raw_room_median_C': raw_room,
            'room_only_offset_C': ROOM_REFERENCE_C - raw_room,
        })

hx_channel_room_offsets = pd.DataFrame(channel_offset_rows)
hx_logged_label_room_offsets = pd.DataFrame(logged_label_offset_rows)

print(f'Room reference: {ROOM_REFERENCE_F:.1f} F = {ROOM_REFERENCE_C:.3f} C')
print('Channel-based HX room offsets:')
display(
    hx_channel_room_offsets.style.format({
        'raw_room_median_C': '{:.3f}',
        'room_only_offset_C': '{:+.3f}',
    }).hide(axis='index')
)

print('Logged-label HX room offsets from the same windows:')
display(
    hx_logged_label_room_offsets.style.format({
        'raw_room_median_C': '{:.3f}',
        'room_only_offset_C': '{:+.3f}',
    }).hide(axis='index')
)

summary_rows = []
smoke_rows = []
for path in sorted((repo_root / 'data' / 'raw').rglob('*.csv')):
    loaded = read_hx_channel_first_csv(path)
    if loaded is None:
        continue
    frame, metadata = loaded
    u8_valid_fraction = float(frame['U8_HX_C'].notna().mean())
    u9_valid_fraction = float(frame['U9_HX_C'].notna().mean())
    smoke_rows.append({
        'path': metadata['relative_path'],
        'header_mapping': metadata['header_mapping'],
        'u8_valid_fraction': u8_valid_fraction,
        'u9_valid_fraction': u9_valid_fraction,
    })
    if u8_valid_fraction < 0.05 or u9_valid_fraction < 0.05:
        continue

    u8_min, u8_min_time = _finite_min_info(frame, 'U8_HX_C')
    u9_min, u9_min_time = _finite_min_info(frame, 'U9_HX_C')
    u8_first_drop = _first_drop_time_min(frame, 'U8_HX_C')
    u9_first_drop = _first_drop_time_min(frame, 'U9_HX_C')
    tc_note = 'yes' if metadata['tc_calibrated'] else 'no'
    if metadata['calibration_file']:
        tc_note = f"yes ({metadata['calibration_file']})"

    summary_rows.append({
        'path': metadata['relative_path'],
        'samples': int(len(frame)),
        'duration_min': float(frame['t_rel_min'].iloc[-1]),
        'header_mapping': metadata['header_mapping'],
        'tc_calibrated_log': tc_note,
        'U8_start_C': float(frame['U8_HX_C'].dropna().iloc[0]),
        'U9_start_C': float(frame['U9_HX_C'].dropna().iloc[0]),
        'U8_min_C': u8_min,
        'U8_min_time_min': u8_min_time,
        'U9_min_C': u9_min,
        'U9_min_time_min': u9_min_time,
        'U8_minus_U9_min_C': u8_min - u9_min,
        'colder_channel_by_minimum': 'U8' if u8_min < u9_min else 'U9',
        'U8_first_5C_drop_min': u8_first_drop,
        'U9_first_5C_drop_min': u9_first_drop,
        'first_5C_response_channel': _first_response_channel(u8_first_drop, u9_first_drop),
    })

hx_channel_smoke = pd.DataFrame(smoke_rows)
hx_cross_log_all = pd.DataFrame(summary_rows).sort_values('path').reset_index(drop=True)

important_paths = {
    'data/raw/calibration/log_20260410_145703.csv',
    'data/raw/calibration/log_20260420_111545.csv',
    'data/raw/recirculation/log_20260424_153546.csv',
    'data/raw/log_20260506_190814.csv',
}
if hx_cross_log_all.empty:
    hx_cross_log_evidence = hx_cross_log_all.copy()
else:
    informative_mask = (
        hx_cross_log_all['U8_min_C'].le(HX_INFORMATIVE_MIN_C)
        | hx_cross_log_all['U9_min_C'].le(HX_INFORMATIVE_MIN_C)
        | hx_cross_log_all['path'].isin(important_paths)
    )
    hx_cross_log_evidence = hx_cross_log_all.loc[informative_mask].reset_index(drop=True)

print('Smoke check for channel-first mapping:')
display(pd.DataFrame([{
    'raw_csv_files_seen': len(list((repo_root / 'data' / 'raw').rglob('*.csv'))),
    'ten_channel_temperature_logs_mapped': len(hx_channel_smoke),
    'logs_with_populated_U8_and_U9': len(hx_cross_log_all),
}]))

print('Header mapping counts across mapped raw logs:')
display(hx_channel_smoke['header_mapping'].value_counts().rename_axis('header_mapping').reset_index(name='count'))

print('Channel-first cold/coupling evidence from informative logs:')
display(
    hx_cross_log_evidence.style.format({
        'duration_min': '{:.2f}',
        'U8_start_C': '{:.2f}',
        'U9_start_C': '{:.2f}',
        'U8_min_C': '{:.2f}',
        'U8_min_time_min': '{:.2f}',
        'U9_min_C': '{:.2f}',
        'U9_min_time_min': '{:.2f}',
        'U8_minus_U9_min_C': '{:+.2f}',
        'U8_first_5C_drop_min': '{:.2f}',
        'U9_first_5C_drop_min': '{:.2f}',
    }).hide(axis='index')
)


Room reference: 68.5 F = 20.278 C
Channel-based HX room offsets:


run,current_reference,hardware_channel,source_column_in_file,room_window,raw_room_median_C,room_only_offset_C
Apr 10 original calibration,False,U8,THI_C,pre-dip fixed window,20.480,-0.202
Apr 10 original calibration,False,U9,THM_C,pre-dip fixed window,20.520,-0.242
Apr 20 corrected-type calibration,True,U8,THI_C,"flat tail, last 300 s",20.920,-0.642
Apr 20 corrected-type calibration,True,U9,THM_C,"flat tail, last 300 s",19.690,+0.588


Logged-label HX room offsets from the same windows:


run,current_reference,logged_label,hardware_channel_in_this_file,raw_room_median_C,room_only_offset_C
Apr 10 original calibration,False,THI,U8,20.480,-0.202
Apr 10 original calibration,False,THM,U9,20.520,-0.242
Apr 20 corrected-type calibration,True,THI,U8,20.920,-0.642
Apr 20 corrected-type calibration,True,THM,U9,19.690,+0.588


Smoke check for channel-first mapping:


,raw_csv_files_seen,ten_channel_temperature_logs_mapped,logs_with_populated_U8_and_U9
0,55,55,53


Header mapping counts across mapped raw logs:


,header_mapping,count
0,"U8->THI_C, U9->THM_C",27
1,"U8->temp8_C, U9->temp9_C",16
2,"U8->THM_C, U9->THI_C",12


Channel-first cold/coupling evidence from informative logs:


path,samples,duration_min,header_mapping,tc_calibrated_log,U8_start_C,U9_start_C,U8_min_C,U8_min_time_min,U9_min_C,U9_min_time_min,U8_minus_U9_min_C,colder_channel_by_minimum,U8_first_5C_drop_min,U9_first_5C_drop_min,first_5C_response_channel
data/raw/LN_dip/log_20260423_105420.csv,2551,88.47,"U8->THI_C, U9->THM_C",yes (TC_calibration_20260420.csv),2.56,4.27,2.53,0.31,4.14,0.28,-1.61,U8,nan,nan,none
data/raw/archive/_pump_log_backups/log_20260402_150754.csv,4793,142.53,"U8->temp8_C, U9->temp9_C",no,20.05,20.02,5.98,107.04,4.84,107.01,+1.14,U9,95.72,94.03,U9
data/raw/archive/_pump_log_backups/log_20260403_115916.csv,2587,78.12,"U8->temp8_C, U9->temp9_C",no,19.97,20.05,-4.52,62.38,-5.11,62.17,+0.59,U9,27.86,27.67,U9
data/raw/archive/_pump_log_backups/log_20260417_094053.csv,10091,305.84,"U8->THI_C, U9->THM_C",no,19.91,19.86,-55.67,127.52,-111.98,95.10,+56.31,U9,23.88,23.82,U9
data/raw/archive/tc_calibrated_20260420_originals/LN_dip/log_20260423_105420.csv,2551,88.47,"U8->THI_C, U9->THM_C",yes (TC_calibration_20260420.csv),3.15,3.63,3.12,0.31,3.50,0.28,-0.38,U8,nan,nan,none
data/raw/archive/tc_calibrated_20260420_originals/log_20260506_190814.csv,785,28.52,"U8->THI_C, U9->THM_C",yes (TC_calibration_20260420.csv),19.24,20.50,-153.84,17.50,-101.78,17.71,-52.06,U8,5.75,2.69,U9
data/raw/archive/tc_calibrated_20260420_originals/recirculation/log_20260422_143345.csv,11210,388.61,"U8->THI_C, U9->THM_C",yes (TC_calibration_20260420.csv),20.17,21.37,-114.99,56.86,-62.00,82.09,-52.99,U8,5.31,5.06,U9
data/raw/archive/tc_calibrated_20260420_originals/recirculation/log_20260424_153546.csv,9467,330.04,"U8->THI_C, U9->THM_C",yes (TC_calibration_20260420.csv),20.91,22.15,-142.11,83.12,-99.58,140.24,-42.53,U8,2.16,2.27,U8
data/raw/archive/u8_u9_header_originals/recirculation/log_20260417_094053.csv,10091,305.84,"U8->THM_C, U9->THI_C",no,19.91,19.86,-55.67,127.52,-111.98,95.10,+56.31,U9,23.88,23.82,U9
data/raw/calibration/log_20260410_145703.csv,3591,108.65,"U8->THI_C, U9->THM_C",no,20.45,20.54,-172.32,102.14,-146.92,102.32,-25.40,U8,94.14,93.75,U9


## Channel-first THI/THM diagnostic plots

These plots intentionally separate calibration anchors from diagnostics. For the installed HX probes, the only hard anchor available is the room-temperature reference. The cold data points are plotted as multi-log evidence for channel identity and cold coupling, not as multipoint absolute calibration points.


In [25]:
import matplotlib.pyplot as plt

required_channel_first_objects = [
    'hx_channel_room_offsets',
    'hx_cross_log_evidence',
    'ROOM_REFERENCE_C',
]
missing = [name for name in required_channel_first_objects if name not in globals()]
if missing:
    raise RuntimeError(f"Run the channel-first diagnosis cell first; missing: {', '.join(missing)}")

room_plot = hx_channel_room_offsets.copy()
evidence_plot = hx_cross_log_evidence.copy()

channel_colors = {'U8': '#dc2626', 'U9': '#2563eb'}
header_markers = {
    'U8->THI_C, U9->THM_C': 'o',
    'U8->THM_C, U9->THI_C': 's',
    'U8->temp8_C, U9->temp9_C': '^',
}

def _short_log_label(path: str) -> str:
    stem = Path(path).stem
    if stem.startswith('log_'):
        stem = stem.removeprefix('log_')
    parent = Path(path).parent.name
    if parent in {'calibration', 'recirculation', 'LN_dip'}:
        return f'{parent}/{stem}'
    return stem

fig, axes = plt.subplots(1, 3, figsize=(17.0, 5.0), constrained_layout=True)

ax = axes[0]
room_plot = room_plot.sort_values(['current_reference', 'run', 'hardware_channel']).reset_index(drop=True)
x_positions = np.arange(len(room_plot))
for idx, row in room_plot.iterrows():
    channel = row['hardware_channel']
    color = channel_colors.get(channel, '0.35')
    raw_room = row['raw_room_median_C']
    ax.vlines(idx, raw_room, ROOM_REFERENCE_C, color=color, lw=2.0, alpha=0.65)
    ax.scatter(idx, raw_room, color=color, edgecolor='white', s=65, zorder=3)
    ax.scatter(idx, ROOM_REFERENCE_C, color=color, marker='D', edgecolor='white', s=55, zorder=4)
    ax.text(
        idx,
        max(raw_room, ROOM_REFERENCE_C) + 0.08,
        f"{channel}\n{row['source_column_in_file']}\n{row['room_only_offset_C']:+.3f} C",
        ha='center',
        va='bottom',
        fontsize=8,
    )
ax.axhline(ROOM_REFERENCE_C, color='0.2', ls='--', lw=1.0, label=f'room reference ({ROOM_REFERENCE_C:.3f} C)')
ax.set_xticks(x_positions)
ax.set_xticklabels(room_plot['run'].str.replace(' calibration', '', regex=False), rotation=25, ha='right')
ax.set_ylabel('Temperature [C]')
ax.set_title('Hard anchor: room-only HX offsets')
ax.grid(True, axis='y', alpha=0.25)
ax.legend(loc='best', fontsize=8)

ax = axes[1]
min_low = float(np.nanmin([evidence_plot['U8_min_C'].min(), evidence_plot['U9_min_C'].min(), -170.0])) - 5.0
min_high = float(np.nanmax([evidence_plot['U8_min_C'].max(), evidence_plot['U9_min_C'].max(), 25.0])) + 5.0
for mapping, group in evidence_plot.groupby('header_mapping'):
    marker = header_markers.get(mapping, 'o')
    ax.scatter(
        group['U8_min_C'],
        group['U9_min_C'],
        marker=marker,
        s=64,
        edgecolor='white',
        alpha=0.9,
        label=mapping,
    )
    for _, row in group.iterrows():
        ax.annotate(
            _short_log_label(row['path']),
            (row['U8_min_C'], row['U9_min_C']),
            xytext=(4, 4),
            textcoords='offset points',
            fontsize=7,
            alpha=0.82,
        )
ax.plot([min_low, min_high], [min_low, min_high], color='0.35', ls='--', lw=1.0, label='U8 = U9')
ax.set_xlim(min_low, min_high)
ax.set_ylim(min_low, min_high)
ax.set_xlabel('U8 minimum [C]')
ax.set_ylabel('U9 minimum [C]')
ax.set_title('Cold diagnostics: minimum by hardware channel')
ax.grid(True, alpha=0.25)
ax.legend(loc='best', fontsize=7)

ax = axes[2]
time_plot = evidence_plot.dropna(subset=['U8_first_5C_drop_min', 'U9_first_5C_drop_min']).copy()
if not time_plot.empty:
    t_high = float(np.nanmax([time_plot['U8_first_5C_drop_min'].max(), time_plot['U9_first_5C_drop_min'].max()])) + 5.0
    t_low = 0.0
    for mapping, group in time_plot.groupby('header_mapping'):
        marker = header_markers.get(mapping, 'o')
        ax.scatter(
            group['U8_first_5C_drop_min'],
            group['U9_first_5C_drop_min'],
            marker=marker,
            s=64,
            edgecolor='white',
            alpha=0.9,
            label=mapping,
        )
        for _, row in group.iterrows():
            ax.annotate(
                _short_log_label(row['path']),
                (row['U8_first_5C_drop_min'], row['U9_first_5C_drop_min']),
                xytext=(4, 4),
                textcoords='offset points',
                fontsize=7,
                alpha=0.82,
            )
    ax.plot([t_low, t_high], [t_low, t_high], color='0.35', ls='--', lw=1.0, label='same response time')
    ax.set_xlim(t_low, t_high)
    ax.set_ylim(t_low, t_high)
else:
    ax.text(0.5, 0.5, 'No logs with both channels crossing a 5 C drop', ha='center', va='center')
ax.set_xlabel('U8 first 5 C drop [min]')
ax.set_ylabel('U9 first 5 C drop [min]')
ax.set_title('Cold diagnostics: first response timing')
ax.grid(True, alpha=0.25)

plt.show()



/tmp/ipykernel_1222632/2057358720.py:126: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Interpretation

This channel-first view changes the safe conclusion:

- `THI_C` and `THM_C` are not reliable historical identifiers by themselves. The CSV header order changed across runs, so any cross-run comparison must first normalize the ninth and tenth thermocouple slots to `U8_HX_C` and `U9_HX_C`.
- The April 20 flat-tail room window is the current hard calibration reference. It provides room-only offsets by channel, and the table reports those separately from the logged-label offsets. This section intentionally does not export or overwrite `TC_calibration_20260420.csv`.
- There is still no defensible low-temperature hard anchor for either installed HX probe. The April 10 HX dip was partial, the April 24 HFE freezing/icing feature is a system behavior rather than a known probe equilibrium, and the May 6 nitrogen-gas test has no fixed local phase-transition temperature at the probe tips.
- The cold logs are still useful: they rank effective cold coupling and reveal label/channel inconsistencies. They do not determine the true coil-metal temperature.
- The geometry note matters physically. A probe around 0.3 cm from the coil metal and another at least 1 cm away will mix coil-metal temperature, gas or HFE temperature, contact resistance, and probe time constant. Inferring coil-metal temperature from those offsets requires a separate thermal model or an independent reference measurement near the coil.